<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/BENCHMARK_TOPOLOGICAL_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AST_LEFM standing for Arithmetic Spectral Theory - Laplace Euler Fourier Mellin.

These pieces combine into the L-EFM operator tested in the notebook, it acts as a rigid mathematical filter. It explains why the "Spectral Trap" drastically rejects values outside of $\sigma = 0.5$ due to massive error amplification, while establishing the hard boundary constant ($\Lambda$) used for the geometric governance and AI safety vector filtering.It is a beautiful synthesis of classical analytic number theory and modern operator physics.

AST/L-EFM - A Complete Python Library for Spectral Quantification of Prime Theorems, Proof of the Riemann Hypothesis, and Deterministic AI Safety: https://zenodo.org/records/20275803

In [ ]:
!nvidia-smi

Sat May 23 11:17:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## SETUP

In [ ]:
!git clone https://github.com/frank-morales2020/ast_lefm.git

Cloning into 'ast_lefm'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 43 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 20.91 KiB | 2.61 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [ ]:
!ls -ltha /content/ast_lefm/

total 60K
drwxr-xr-x 4 root root 4.0K May 23 11:17 .
-rw-r--r-- 1 root root  970 May 23 11:17 chowla.py
-rw-r--r-- 1 root root 1.1K May 23 11:17 constants.py
-rw-r--r-- 1 root root 2.6K May 23 11:17 einstein.py
drwxr-xr-x 8 root root 4.0K May 23 11:17 .git
-rw-r--r-- 1 root root 1.5K May 23 11:17 gtt_decay.py
-rw-r--r-- 1 root root 2.0K May 23 11:17 h2e.py
-rw-r--r-- 1 root root 1.1K May 23 11:17 __init__.py
-rw-r--r-- 1 root root 3.2K May 23 11:17 lefm.py
-rw-r--r-- 1 root root 5.3K May 23 11:17 README.md
-rw-r--r-- 1 root root  911 May 23 11:17 setup.py
-rw-r--r-- 1 root root 1.9K May 23 11:17 sieve.py
drwxr-xr-x 2 root root 4.0K May 23 11:17 test
drwxr-xr-x 1 root root 4.0K May 23 11:17 ..


In [ ]:
!ls -ltha /content/ast_lefm/test/

total 12K
drwxr-xr-x 2 root root 4.0K May 23 11:17 .
drwxr-xr-x 4 root root 4.0K May 23 11:17 ..
-rw-r--r-- 1 root root 2.2K May 23 11:17 test_all.py


In [ ]:
%cd /content/ast_lefm
!pip install -e .

/content/ast_lefm
Obtaining file:///content/ast_lefm
  Preparing metadata (setup.py) ... done
  Running setup.py develop for ast_lefm


In [ ]:
import sys
sys.path.insert(0, '/content/ast_lefm')

In [ ]:
import sys
import os

# Force add the path
sys.path.insert(0, '/content/ast_lefm')
sys.path.insert(0, '/content/ast_lefm/ast_lefm')

# Try to import
try:
    from ast_lefm.sieve import primes_up_to
    print("✓ Import successful")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    # Let's see what's in the directory
    print("\nFiles in /content/ast_lefm:")
    os.listdir('/content/ast_lefm')

    print("\nTrying alternative import...")
    # Try importing as a local module
    import importlib.util
    spec = importlib.util.spec_from_file_location("sieve", "/content/ast_lefm/sieve.py")
    sieve = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(sieve)
    primes_up_to = sieve.primes_up_to
    print("✓ Alternative import worked")

✓ Import successful


In [ ]:
!pip install --upgrade bitsandbytes accelerate transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 210.3 MB/s eta 0:00:00


In [ ]:
import torch
import bitsandbytes as bnb
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'bitsandbytes: {bnb.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.10.0+cu128
CUDA: 12.8
bitsandbytes: 0.49.2
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## TWO-TASK BENCHMARK (3 EPOCHS)

In [ ]:
# ============================================================================
# CORRECTED COMPLETE COMPARISON: Topological AI vs Existing Methods
#
# Fixes vs notebook version:
#
# 1. SEPARATE CLASSIFIERS (TaskAwareModel with classifier_A + classifier_B).
#    The shared-head design let enforce_anchors() freeze Task B entirely,
#    making Topological's 42% Task B / -0.7% forgetting a construction
#    artifact, not a real result. Separate heads eliminate this.
#
# 2. FAIR ARCHITECTURE: All methods share identical model structure.
#    classifier_A frozen via requires_grad=False after Task A in every
#    method. The embedding layer is the only shared, trainable component
#    that can cause cross-task interference.
#
# 3. TOPOLOGICAL: enforce_anchors() restores EMBEDDING ROWS ONLY.
#    classifier_A is protected by requires_grad=False, not overwriting.
#
# 4. REAL EWC: Empirical Fisher Information diagonal on the embedding
#    layer. Consistent EWC_LAMBDA=5000 across all experiments (was 500
#    in notebook, 10× weaker than all other cells — artificially hobbled).
#
# 5. CORRECT REPLAY: Task A samples routed through classifier_A,
#    Task B samples routed through classifier_B. Single backward() call
#    over combined loss lets both tasks pull the shared embedding.
#
# 6. HOPE-like (Dual Timescale EMA): Honest dual-timescale implementation.
#    A slow EMA of the embedding tracks post-Task-A state. A penalty
#    during Task B pulls the fast (optimizer) weights toward the slow EMA.
#    EMA target drifts gradually unlike EWC's fixed snapshot, giving the
#    embedding some Task B plasticity while resisting large drift.
#
# 7. evaluate() is side-effect-free: restores model.training and
#    model.current_task before returning.
#
# 8. Per-run deterministic seeding (SEED + run) for independent runs.
#
# 9. Mean ± std reported over N_RUNS.
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc
import random
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from warnings import filterwarnings
filterwarnings('ignore')

import sys
sys.path.insert(0, '/content/ast_lefm')
try:
    from ast_lefm.sieve import primes_up_to
except ImportError:
    def primes_up_to(n):
        """Fallback sieve of Eratosthenes."""
        sieve = [True] * (n + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(n ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, n + 1, i):
                    sieve[j] = False
        return [i for i in range(2, n + 1) if sieve[i]]


# ============================================================
# Global config
# ============================================================
SEED         = 123
N_RUNS       = 5       # mean ± std over this many independent runs
BATCH_SIZE   = 16
REPLAY_HALF  = 8       # samples per task in mixed replay batches
MAX_LEN      = 64
EPOCHS       = 3
LR_EMBED     = 5e-3    # raised from 1e-4 — produces measurable embedding drift and forgetting
LR_CLS       = 1e-3
EWC_LAMBDA   = 5000    # consistent across ALL methods (was 500 in notebook)
EMA_BETA     = 0.9     # HOPE-like slow EMA decay
EMA_LAMBDA   = 5000    # HOPE-like penalty weight (matched to EWC for fairness)
PRIME_LIMIT  = 13      # matches paper: [2, 3, 5, 7, 11, 13] → 6 anchor rows
BUFFER_SIZE  = 200
MODEL_NAME   = "openai/gpt-oss-20b"


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print("=" * 80)
print("CORRECTED COMPARISON: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline")
print("=" * 80)


# =====================================================================
# 1. DATASET
# =====================================================================
dataset = load_dataset("ag_news", split="train")


def create_task(dataset, class_labels, num_samples=500):
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    sampled  = filtered.select(range(min(num_samples, len(filtered))))
    texts    = [item['text']       for item in sampled]
    labels   = [item['label'] % 2  for item in sampled]   # binary within task
    return texts, labels


task_a_texts, task_a_labels = create_task(dataset, [0, 1], 500)
#task_b_texts, task_b_labels = create_task(dataset, [2, 3], 500)
task_b_texts, task_b_labels = create_task(dataset, [2, 3], 1000)

print(f"Task A: {len(task_a_texts)} samples  (World=0, Sports=1)")
print(f"Task B: {len(task_b_texts)} samples  (Business=0, Sci/Tech=1)")


# =====================================================================
# 2. MODEL — SEPARATE CLASSIFIER HEAD PER TASK
#
# Why this matters: the notebook's SharedClassifierModel let
# enforce_anchors() reset the single classifier head after every Task B
# gradient step, preventing Task B from training at all and producing
# a fraudulent -0.7% forgetting result. Separate heads make the heads
# structurally independent: only the shared embedding layer can transfer
# interference between tasks.
# =====================================================================
class TaskAwareModel(nn.Module):
    def __init__(self, base_model, hidden_size=2880):
        super().__init__()
        self.base_model   = base_model
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs     = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        last_hidden = outputs.hidden_states[-1][:, -1, :]
        head        = self.classifier_A if self.current_task == 'A' else self.classifier_B
        return head(last_hidden), outputs

    def switch_task(self, task: str):
        assert task in ('A', 'B'), f"Unknown task: {task}"
        self.current_task = task

    def freeze_classifier_a(self):
        """
        Explicitly freeze classifier_A after Task A training.
        NOTE: requires_grad=False on the weight tensor does NOT prevent
        gradients from flowing THROUGH classifier_A into the shared
        embedding layer. This is intentional for Replay: Task A replay
        samples still contribute embedding gradients.
        """
        self.classifier_A.requires_grad_(False)


# =====================================================================
# 3. SHARED UTILITIES
# =====================================================================
def get_embed_layer(base_model: nn.Module) -> nn.Embedding:
    """Locate the token embedding layer regardless of model family."""
    if hasattr(base_model, 'transformer') and hasattr(base_model.transformer, 'wte'):
        return base_model.transformer.wte
    for module in base_model.modules():
        if isinstance(module, nn.Embedding) and module.weight.shape[0] > 100_000:
            return module
    raise RuntimeError("Could not locate embedding layer.")


@torch.no_grad()
def evaluate(model, tokenizer, texts, labels, task: str, batch_size: int = 32) -> float:
    """
    Pure evaluation — no side effects.
    Saves and restores both model.training state and model.current_task.
    """
    was_training  = model.training
    previous_task = model.current_task
    model.eval()
    model.switch_task(task)

    all_preds, all_labels = [], []
    for i in range(0, len(texts), batch_size):
        tokens = tokenizer(
            texts[i:i + batch_size], return_tensors="pt",
            padding=True, truncation=True, max_length=MAX_LEN,
        ).to(device)
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels[i:i + batch_size])

    model.switch_task(previous_task)
    if was_training:
        model.train()
    return accuracy_score(all_labels, all_preds)


def tokenize(tokenizer, texts, labels):
    tokens = tokenizer(
        texts, return_tensors="pt",
        padding=True, truncation=True, max_length=MAX_LEN,
    ).to(device)
    return tokens, torch.tensor(labels, dtype=torch.long).to(device)


def load_fresh_model():
    """Load a clean model copy. Backbone is immediately frozen."""
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = TaskAwareModel(base_model).to(device)
    for param in base_model.parameters():
        param.requires_grad = False
    return model, tokenizer, base_model


def train_task_a(model, tokenizer, optimizer):
    """Shared Task A loop — identical across all methods."""
    model.switch_task('A')
    model.train()
    for _ in range(EPOCHS):
        for i in range(0, len(task_a_texts), BATCH_SIZE):
            tokens, labels = tokenize(
                tokenizer,
                task_a_texts[i:i + BATCH_SIZE],
                task_a_labels[i:i + BATCH_SIZE],
            )
            optimizer.zero_grad()
            logits, _ = model(tokens.input_ids, tokens.attention_mask)
            F.cross_entropy(logits, labels).backward()
            optimizer.step()


def build_task_b_optimizer(embed_layer, model):
    """
    Task B optimizer: embedding + classifier_B only.
    classifier_A is already frozen and absent from this optimizer.
    """
    return bnb.optim.AdamW8bit([
        {'params': embed_layer.weight,               'lr': LR_EMBED},
        {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
    ])


def task_b_step(loss, embed_layer, optimizer, max_norm=1.0):
    """
    Shared Task B gradient step applied identically across ALL methods.
    Gradient clipping on the embedding prevents seed-specific gradient
    spikes at LR_EMBED=5e-3 from destabilising training. Applied to
    every method for a fair comparison — no method gets a stability
    advantage the others do not have.
    """
    loss.backward()
    torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=max_norm)
    optimizer.step()


def summarise(results: list) -> dict:
    fgt = [r['forgetting']   for r in results]
    ab  = [r['acc_b']        for r in results]
    aa  = [r['acc_a_final']  for r in results]
    return {
        'forgetting_mean':   np.mean(fgt),  'forgetting_std':   np.std(fgt),
        'acc_b_mean':        np.mean(ab),   'acc_b_std':        np.std(ab),
        'acc_a_final_mean':  np.mean(aa),   'acc_a_final_std':  np.std(aa),
    }


def log_run(acc_a_initial, acc_a_final, acc_b) -> float:
    forgetting = (acc_a_initial - acc_a_final) * 100
    print(f"    Task A initial: {acc_a_initial * 100:.1f}%")
    print(f"    Task A final:   {acc_a_final * 100:.1f}%  |  Forgetting: {forgetting:+.1f}%")
    print(f"    Task B acc:     {acc_b * 100:.1f}%")
    return forgetting


def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


# =====================================================================
# 4. TOPOLOGICAL GOVERNOR
#
# Restores a sparse set of prime-indexed embedding rows after each
# Task B optimizer step. This is the ONLY thing enforce_anchors() does.
# classifier_A is protected by freeze_classifier_a() (requires_grad=False),
# NOT by weight overwriting — fixing the core notebook flaw.
#
# OPTIMIZER STATE RESET: After hard-restoring anchor rows, the AdamW8bit
# first and second moment estimates for those rows must also be zeroed.
# Without this, the optimizer accumulates momentum as if the rows moved
# freely, creating a growing mismatch between optimizer state and actual
# weights. At LR_EMBED=5e-3 this mismatch is large enough to destabilise
# training in some runs (observed: +46% forgetting in Run 2 without fix).
# =====================================================================
class TopologicalGovernor:
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer    = embed_layer
        vocab_size          = embed_layer.weight.shape[0]
        self.anchor_indices = [p for p in primes_up_to(prime_limit) if p < vocab_size]
        self.snapshot: dict = {}
        print(f"  → Anchoring {len(self.anchor_indices)} prime rows: {self.anchor_indices}")

    def take_snapshot(self):
        """Call immediately after Task A training completes."""
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def enforce_anchors(self):
        """Restore prime-indexed rows. Called after every Task B optimizer.step()."""
        if not self.snapshot:
            raise RuntimeError("Call take_snapshot() before enforce_anchors().")
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    @torch.no_grad()
    def zero_anchor_gradients(self):
        """
        Zero gradients for anchor rows BEFORE optimizer.step().

        This is the correct fix for bitsandbytes AdamW8bit compatibility.
        bnb uses internal state keys 'state1'/'state2' with block-wise
        quantization maps — not the standard PyTorch 'exp_avg'/'exp_avg_sq'
        keys. Trying to reset bnb state directly is fragile and optimizer-
        specific.

        Zeroing the gradient at source is optimizer-agnostic and solves the
        problem at the right level: if the gradient for an anchor row is zero
        before the optimizer step, the optimizer computes no update for that
        row, accumulates no momentum, and the weights remain at their anchored
        values without needing enforce_anchors() to correct them afterward.
        enforce_anchors() is kept as a safety assertion only.
        """
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )


# =====================================================================
# 5. FISHER DIAGONAL  (for EWC)
# =====================================================================
def compute_fisher(model, tokenizer, embed_layer, n_samples: int = 200) -> torch.Tensor:
    """
    Empirical Fisher Information diagonal on the embedding weight matrix.
    Estimated on Task A data immediately after Task A training.
    Returns shape (vocab_size, hidden_size).
    """
    model.eval()
    model.switch_task('A')
    fisher    = torch.zeros_like(embed_layer.weight, dtype=torch.float32)
    n_batches = 0

    for i in range(0, min(n_samples, len(task_a_texts)), BATCH_SIZE):
        tokens, labels = tokenize(
            tokenizer,
            task_a_texts[i:i + BATCH_SIZE],
            task_a_labels[i:i + BATCH_SIZE],
        )
        model.zero_grad()
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        F.cross_entropy(logits, labels).backward()
        if embed_layer.weight.grad is not None:
            fisher    += embed_layer.weight.grad.float() ** 2
            n_batches += 1

    fisher /= max(n_batches, 1)
    model.zero_grad()
    return fisher


# =====================================================================
# 6. DUAL TIMESCALE EMA  (for HOPE-like method)
#
# Honest dual-timescale implementation:
# - A slow EMA of the embedding tracks the post-Task-A embedding state.
# - During Task B, a penalty term pulls the fast (optimizer) weights
#   toward the slow EMA target.
# - After each optimizer step, the slow EMA is updated with decay beta,
#   allowing it to drift gradually rather than being frozen (unlike EWC).
#
# This differs meaningfully from EWC:
#   EWC:       fixed Task A snapshot, Fisher-weighted per-dimension penalty
#   HOPE-like: dynamic EMA target, uniform-weighted penalty
#
# Both use EMA_LAMBDA = EWC_LAMBDA for a fair penalty-strength comparison.
# =====================================================================
class DualTimescaleEMA:
    def __init__(self, embed_layer: nn.Embedding,
                 beta: float = EMA_BETA, lambda_penalty: float = EMA_LAMBDA):
        self.embed_layer    = embed_layer
        self.beta           = beta
        self.lambda_penalty = lambda_penalty
        # Slow EMA initialised to post-Task-A embedding
        self.slow_embed     = embed_layer.weight.detach().clone().float()

    @torch.no_grad()
    def update_slow(self):
        """EMA step — called after every Task B optimizer.step()."""
        current    = self.embed_layer.weight.float()
        self.slow_embed = self.beta * self.slow_embed + (1.0 - self.beta) * current

    def penalty(self) -> torch.Tensor:
        """L2 pull of fast weights toward slow EMA target."""
        delta = self.embed_layer.weight.float() - self.slow_embed
        return self.lambda_penalty * (delta ** 2).sum()


# =====================================================================
# 7. BASELINE — shared embedding, no protection
# =====================================================================
def train_baseline():
    print(f"\n{'=' * 60}")
    print("Method: BASELINE  (shared embedding, no protection)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')

        # Task B — embedding drifts freely
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_b_step(F.cross_entropy(logits, labels), embed_layer, opt_b)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b       = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting  = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 8. TOPOLOGICAL AI — prime embedding row anchoring
# =====================================================================
def train_topological():
    print(f"\n{'=' * 60}")
    print("Method: TOPOLOGICAL AI  (prime embedding anchors)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        # Snapshot prime rows AFTER Task A completes
        governor = TopologicalGovernor(embed_layer)
        governor.take_snapshot()
        model.freeze_classifier_a()   # explicit freeze — not covert overwriting

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')

        # Task B — restore prime rows after each step
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()              # zero prime row gradients before clip and step
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)  # same clip as all other methods
                opt_b.step()
                governor.enforce_anchors()                    # safety assertion — weights already unchanged

        assert governor.verify_integrity(), "Anchor integrity check failed!"

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b       = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting  = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 9. EWC — empirical Fisher Information diagonal on embedding layer
# =====================================================================
def train_ewc():
    print(f"\n{'=' * 60}")
    print("Method: EWC  (Fisher Information diagonal on embedding)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        # Fisher diagonal + fixed Task A embedding snapshot
        fisher         = compute_fisher(model, tokenizer, embed_layer)
        embed_snapshot = embed_layer.weight.detach().clone().float()
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')

        # Task B with Fisher-weighted EWC penalty on embedding drift
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss  = F.cross_entropy(logits, labels)
                # EWC: Σ F_i · (θ_i − θ*_i)²
                delta      = embed_layer.weight.float() - embed_snapshot
                ewc_loss   = (fisher * delta ** 2).sum()
                task_b_step(task_loss + EWC_LAMBDA * ewc_loss, embed_layer, opt_b)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b       = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting  = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model, fisher, embed_snapshot)

    return summarise(results)


# =====================================================================
# 10. REPLAY — Task A samples correctly routed through classifier_A
#
# Key fix: Task A replay samples go through classifier_A, Task B samples
# through classifier_B. A single backward() call over the combined loss
# lets both tasks contribute gradients to the shared embedding in one pass.
#
# With classifier_A frozen (requires_grad=False), the Task A replay loss
# does not update classifier_A's weights, but gradients DO still flow
# back through classifier_A into the embedding layer — this is correct
# and intentional. The embedding receives a Task-A-preserving signal.
# =====================================================================
def train_replay():
    print(f"\n{'=' * 60}")
    print("Method: REPLAY  (Task A via classifier_A, Task B via classifier_B)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        buffer = list(zip(task_a_texts, task_a_labels))[:BUFFER_SIZE]
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                opt_b.zero_grad()

                # ── Task B forward pass ──────────────────────────────
                model.switch_task('B')
                tokens_b, labels_b = tokenize(
                    tokenizer,
                    task_b_texts[i:i + REPLAY_HALF],
                    task_b_labels[i:i + REPLAY_HALF],
                )
                logits_b, _ = model(tokens_b.input_ids, tokens_b.attention_mask)
                loss_b = F.cross_entropy(logits_b, labels_b)

                # ── Task A replay via classifier_A (correct head) ────
                replay_batch       = random.sample(buffer, min(REPLAY_HALF, len(buffer)))
                r_texts, r_labels  = zip(*replay_batch)
                tokens_a, labels_a = tokenize(tokenizer, list(r_texts), list(r_labels))
                model.switch_task('A')
                logits_a, _ = model(tokens_a.input_ids, tokens_a.attention_mask)
                loss_a = F.cross_entropy(logits_a, labels_a)

                # Combined backward: both tasks pull the shared embedding
                task_b_step(loss_b + loss_a, embed_layer, opt_b)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b       = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting  = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 11. HOPE-like — Dual Timescale EMA
# =====================================================================
def train_hope():
    print(f"\n{'=' * 60}")
    print("Method: HOPE-like  (Dual Timescale EMA on embedding)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        # EMA initialised from post-Task-A embedding
        ema = DualTimescaleEMA(embed_layer, beta=EMA_BETA, lambda_penalty=EMA_LAMBDA)
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')

        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _  = model(tokens.input_ids, tokens.attention_mask)
                task_loss  = F.cross_entropy(logits, labels)
                ema_loss   = ema.penalty()        # pulls toward slow EMA target
                task_b_step(task_loss + ema_loss, embed_layer, opt_b)
                ema.update_slow()                 # slow EMA drifts at rate (1 - beta)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b       = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting  = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 12. MAIN
# =====================================================================
if __name__ == "__main__":

    all_results = {
        'Baseline':     train_baseline(),
        'Topological':  train_topological(),
        'EWC':          train_ewc(),
        'Replay':       train_replay(),
        'HOPE-like':    train_hope(),
    }

    print("\n" + "=" * 80)
    print(f"FINAL RESULTS  (mean ± std,  N = {N_RUNS} runs per method)")
    print("=" * 80)

    col = f"{'Method':<16}  {'Forgetting':>14}  {'Task A Final':>14}  {'Task B Acc':>12}"
    print(col)
    print("-" * len(col))
    for method, res in all_results.items():
        print(
            f"{method:<16}"
            f"  {res['forgetting_mean']:>+5.1f}% ±{res['forgetting_std']:>4.1f}"
            f"  {res['acc_a_final_mean'] * 100:>6.1f}% ±{res['acc_a_final_std'] * 100:>4.1f}"
            f"  {res['acc_b_mean'] * 100:>5.1f}% ±{res['acc_b_std'] * 100:>4.1f}"
        )

    print(f"\nRANKING by forgetting (lower = better):")
    ranked = sorted(all_results.items(), key=lambda x: x[1]['forgetting_mean'])
    for i, (method, res) in enumerate(ranked, 1):
        tb = res['acc_b_mean'] * 100
        print(f"  {i}. {method:<16}  {res['forgetting_mean']:>+5.1f}%  |  Task B: {tb:.1f}%")

    print("=" * 80)

Device: cuda
CORRECTED COMPARISON: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Task A: 500 samples  (World=0, Sports=1)
Task B: 1000 samples  (Business=0, Sci/Tech=1)

Method: BASELINE  (shared embedding, no protection)

  Run 1/5


[transformers] MXFP4 quantization requires the `kernels` package: `pip install kernels>=0.12.0`. We will default to dequantizing the model to bf16.


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

...

    Task A initial: 99.0%
    Task A final:   99.0%  |  Forgetting: +0.0%
    Task B acc:     82.5%

  Run 4/5


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A initial: 99.0%
    Task A final:   99.0%  |  Forgetting: +0.0%
    Task B acc:     79.5%

  Run 5/5


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A initial: 99.0%
    Task A final:   99.0%  |  Forgetting: +0.0%
    Task B acc:     80.0%

FINAL RESULTS  (mean ± std,  N = 5 runs per method)
Method                Forgetting    Task A Final    Task B Acc
--------------------------------------------------------------
Baseline           +2.8% ± 1.2    96.3% ± 1.3   96.0% ± 3.6
Topological        +2.5% ± 1.1    96.6% ± 1.3   98.3% ± 1.8
EWC                +2.4% ± 1.7    96.7% ± 1.7   97.0% ± 0.8
Replay             -0.1% ± 0.6    99.2% ± 0.6   82.8% ±10.5
HOPE-like          +0.0% ± 0.0    99.1% ± 0.2   80.4% ± 1.2

RANKING by forgetting (lower = better):
  1. Replay             -0.1%  |  Task B: 82.8%
  2. HOPE-like          +0.0%  |  Task B: 80.4%
  3. EWC                +2.4%  |  Task B: 97.0%
  4. Topological        +2.5%  |  Task B: 98.3%
  5. Baseline           +2.8%  |  Task B: 96.0%


## TWO-TASK BENCHMARK (6 EPOCHS)

In [ ]:
# ============================================================================
# CORRECTED COMPLETE COMPARISON: Topological AI vs Existing Methods
#
# Fixes vs notebook version:
#
# 1. SEPARATE CLASSIFIERS (TaskAwareModel with classifier_A + classifier_B).
#    The shared-head design let enforce_anchors() freeze Task B entirely,
#    making Topological's 42% Task B / -0.7% forgetting a construction
#    artifact, not a real result. Separate heads eliminate this.
#
# 2. FAIR ARCHITECTURE: All methods share identical model structure.
#    classifier_A frozen via requires_grad=False after Task A in every
#    method. The embedding layer is the only shared, trainable component
#    that can cause cross-task interference.
#
# 3. TOPOLOGICAL: enforce_anchors() restores EMBEDDING ROWS ONLY.
#    classifier_A is protected by requires_grad=False, not overwriting.
#
# 4. REAL EWC: Empirical Fisher Information diagonal on the embedding
#    layer. Consistent EWC_LAMBDA=5000 across all experiments (was 500
#    in notebook, 10× weaker than all other cells — artificially hobbled).
#
# 5. CORRECT REPLAY: Task A samples routed through classifier_A,
#    Task B samples routed through classifier_B. Single backward() call
#    over combined loss lets both tasks pull the shared embedding.
#
# 6. HOPE-like (Dual Timescale EMA): Honest dual-timescale implementation.
#    A slow EMA of the embedding tracks post-Task-A state. A penalty
#    during Task B pulls the fast (optimizer) weights toward the slow EMA.
#    EMA target drifts gradually unlike EWC's fixed snapshot, giving the
#    embedding some Task B plasticity while resisting large drift.
#
# 7. evaluate() is side-effect-free: restores model.training and
#    model.current_task before returning.
#
# 8. Per-run deterministic seeding (SEED + run) for independent runs.
#
# 9. Mean ± std reported over N_RUNS.
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc
import random
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from warnings import filterwarnings
filterwarnings('ignore')

import sys
sys.path.insert(0, '/content/ast_lefm')
try:
    from ast_lefm.sieve import primes_up_to
except ImportError:
    def primes_up_to(n):
        """Fallback sieve of Eratosthenes."""
        sieve = [True] * (n + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(n ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, n + 1, i):
                    sieve[j] = False
        return [i for i in range(2, n + 1) if sieve[i]]


# ============================================================
# Global config
# ============================================================
SEED         = 123
N_RUNS       = 5       # mean ± std over this many independent runs
BATCH_SIZE   = 16
REPLAY_HALF  = 8       # samples per task in mixed replay batches
MAX_LEN      = 64
EPOCHS       = 6
LR_EMBED     = 5e-3    # raised from 1e-4 — produces measurable embedding drift and forgetting
LR_CLS       = 1e-3
EWC_LAMBDA   = 5000    # consistent across ALL methods (was 500 in notebook)
EMA_BETA     = 0.9     # HOPE-like slow EMA decay
EMA_LAMBDA   = 5000    # HOPE-like penalty weight (matched to EWC for fairness)
PRIME_LIMIT  = 13      # matches paper: [2, 3, 5, 7, 11, 13] → 6 anchor rows
BUFFER_SIZE  = 200
MODEL_NAME   = "openai/gpt-oss-20b"


EVAL_SIZE_A = 200   # Task A — kept small since it's just the constraint check
EVAL_SIZE_B = 200   # Task B — larger for more reliable measurement of what matters


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print("=" * 80)
print("CORRECTED COMPARISON: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline")
print("=" * 80)


# =====================================================================
# 1. DATASET
# =====================================================================
dataset = load_dataset("ag_news", split="train")


def create_task(dataset, class_labels, num_samples=500):
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    sampled  = filtered.select(range(min(num_samples, len(filtered))))
    texts    = [item['text']       for item in sampled]
    labels   = [item['label'] % 2  for item in sampled]   # binary within task
    return texts, labels


task_a_texts, task_a_labels = create_task(dataset, [0, 1], 500)
#task_b_texts, task_b_labels = create_task(dataset, [2, 3], 500)
task_b_texts, task_b_labels = create_task(dataset, [2, 3], 1000)

print(f"Task A: {len(task_a_texts)} samples  (World=0, Sports=1)")
print(f"Task B: {len(task_b_texts)} samples  (Business=0, Sci/Tech=1)")


# =====================================================================
# 2. MODEL — SEPARATE CLASSIFIER HEAD PER TASK
#
# Why this matters: the notebook's SharedClassifierModel let
# enforce_anchors() reset the single classifier head after every Task B
# gradient step, preventing Task B from training at all and producing
# a fraudulent -0.7% forgetting result. Separate heads make the heads
# structurally independent: only the shared embedding layer can transfer
# interference between tasks.
# =====================================================================
class TaskAwareModel(nn.Module):
    def __init__(self, base_model, hidden_size=2880):
        super().__init__()
        self.base_model   = base_model
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs     = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        last_hidden = outputs.hidden_states[-1][:, -1, :]
        head        = self.classifier_A if self.current_task == 'A' else self.classifier_B
        return head(last_hidden), outputs

    def switch_task(self, task: str):
        assert task in ('A', 'B'), f"Unknown task: {task}"
        self.current_task = task

    def freeze_classifier_a(self):
        """
        Explicitly freeze classifier_A after Task A training.
        NOTE: requires_grad=False on the weight tensor does NOT prevent
        gradients from flowing THROUGH classifier_A into the shared
        embedding layer. This is intentional for Replay: Task A replay
        samples still contribute embedding gradients.
        """
        self.classifier_A.requires_grad_(False)


# =====================================================================
# 3. SHARED UTILITIES
# =====================================================================
def get_embed_layer(base_model: nn.Module) -> nn.Embedding:
    """Locate the token embedding layer regardless of model family."""
    if hasattr(base_model, 'transformer') and hasattr(base_model.transformer, 'wte'):
        return base_model.transformer.wte
    for module in base_model.modules():
        if isinstance(module, nn.Embedding) and module.weight.shape[0] > 100_000:
            return module
    raise RuntimeError("Could not locate embedding layer.")


@torch.no_grad()
def evaluate(model, tokenizer, texts, labels, task: str, batch_size: int = 32) -> float:
    """
    Pure evaluation — no side effects.
    Saves and restores both model.training state and model.current_task.
    """
    was_training  = model.training
    previous_task = model.current_task
    model.eval()
    model.switch_task(task)

    all_preds, all_labels = [], []
    for i in range(0, len(texts), batch_size):
        tokens = tokenizer(
            texts[i:i + batch_size], return_tensors="pt",
            padding=True, truncation=True, max_length=MAX_LEN,
        ).to(device)
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels[i:i + batch_size])

    model.switch_task(previous_task)
    if was_training:
        model.train()
    return accuracy_score(all_labels, all_preds)


def tokenize(tokenizer, texts, labels):
    tokens = tokenizer(
        texts, return_tensors="pt",
        padding=True, truncation=True, max_length=MAX_LEN,
    ).to(device)
    return tokens, torch.tensor(labels, dtype=torch.long).to(device)


def load_fresh_model():
    """Load a clean model copy. Backbone is immediately frozen."""
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = TaskAwareModel(base_model).to(device)
    for param in base_model.parameters():
        param.requires_grad = False
    return model, tokenizer, base_model


def train_task_a(model, tokenizer, optimizer):
    """Shared Task A loop — identical across all methods."""
    model.switch_task('A')
    model.train()
    for _ in range(EPOCHS):
        for i in range(0, len(task_a_texts), BATCH_SIZE):
            tokens, labels = tokenize(
                tokenizer,
                task_a_texts[i:i + BATCH_SIZE],
                task_a_labels[i:i + BATCH_SIZE],
            )
            optimizer.zero_grad()
            logits, _ = model(tokens.input_ids, tokens.attention_mask)
            F.cross_entropy(logits, labels).backward()
            optimizer.step()


def build_task_b_optimizer(embed_layer, model):
    """
    Task B optimizer: embedding + classifier_B only.
    classifier_A is already frozen and absent from this optimizer.
    """
    return bnb.optim.AdamW8bit([
        {'params': embed_layer.weight,               'lr': LR_EMBED},
        {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
    ])


def task_b_step(loss, embed_layer, optimizer, max_norm=1.0):
    """
    Shared Task B gradient step applied identically across ALL methods.
    Gradient clipping on the embedding prevents seed-specific gradient
    spikes at LR_EMBED=5e-3 from destabilising training. Applied to
    every method for a fair comparison — no method gets a stability
    advantage the others do not have.
    """
    loss.backward()
    torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=max_norm)
    optimizer.step()


def summarise(results: list) -> dict:
    fgt = [r['forgetting']   for r in results]
    ab  = [r['acc_b']        for r in results]
    aa  = [r['acc_a_final']  for r in results]
    return {
        'forgetting_mean':   np.mean(fgt),  'forgetting_std':   np.std(fgt),
        'acc_b_mean':        np.mean(ab),   'acc_b_std':        np.std(ab),
        'acc_a_final_mean':  np.mean(aa),   'acc_a_final_std':  np.std(aa),
    }


def log_run(acc_a_initial, acc_a_final, acc_b) -> float:
    forgetting = (acc_a_initial - acc_a_final) * 100
    print(f"    Task A initial: {acc_a_initial * 100:.1f}%")
    print(f"    Task A final:   {acc_a_final * 100:.1f}%  |  Forgetting: {forgetting:+.1f}%")
    print(f"    Task B acc:     {acc_b * 100:.1f}%")
    return forgetting


def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


# =====================================================================
# 4. TOPOLOGICAL GOVERNOR
#
# Restores a sparse set of prime-indexed embedding rows after each
# Task B optimizer step. This is the ONLY thing enforce_anchors() does.
# classifier_A is protected by freeze_classifier_a() (requires_grad=False),
# NOT by weight overwriting — fixing the core notebook flaw.
#
# OPTIMIZER STATE RESET: After hard-restoring anchor rows, the AdamW8bit
# first and second moment estimates for those rows must also be zeroed.
# Without this, the optimizer accumulates momentum as if the rows moved
# freely, creating a growing mismatch between optimizer state and actual
# weights. At LR_EMBED=5e-3 this mismatch is large enough to destabilise
# training in some runs (observed: +46% forgetting in Run 2 without fix).
# =====================================================================
class TopologicalGovernor:
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer    = embed_layer
        vocab_size          = embed_layer.weight.shape[0]
        self.anchor_indices = [p for p in primes_up_to(prime_limit) if p < vocab_size]
        self.snapshot: dict = {}
        print(f"  → Anchoring {len(self.anchor_indices)} prime rows: {self.anchor_indices}")

    def take_snapshot(self):
        """Call immediately after Task A training completes."""
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def enforce_anchors(self):
        """Restore prime-indexed rows. Called after every Task B optimizer.step()."""
        if not self.snapshot:
            raise RuntimeError("Call take_snapshot() before enforce_anchors().")
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    @torch.no_grad()
    def zero_anchor_gradients(self):
        """
        Zero gradients for anchor rows BEFORE optimizer.step().

        This is the correct fix for bitsandbytes AdamW8bit compatibility.
        bnb uses internal state keys 'state1'/'state2' with block-wise
        quantization maps — not the standard PyTorch 'exp_avg'/'exp_avg_sq'
        keys. Trying to reset bnb state directly is fragile and optimizer-
        specific.

        Zeroing the gradient at source is optimizer-agnostic and solves the
        problem at the right level: if the gradient for an anchor row is zero
        before the optimizer step, the optimizer computes no update for that
        row, accumulates no momentum, and the weights remain at their anchored
        values without needing enforce_anchors() to correct them afterward.
        enforce_anchors() is kept as a safety assertion only.
        """
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )


# =====================================================================
# 5. FISHER DIAGONAL  (for EWC)
# =====================================================================
def compute_fisher(model, tokenizer, embed_layer, n_samples: int = 200) -> torch.Tensor:
    """
    Empirical Fisher Information diagonal on the embedding weight matrix.
    Estimated on Task A data immediately after Task A training.
    Returns shape (vocab_size, hidden_size).
    """
    model.eval()
    model.switch_task('A')
    fisher    = torch.zeros_like(embed_layer.weight, dtype=torch.float32)
    n_batches = 0

    for i in range(0, min(n_samples, len(task_a_texts)), BATCH_SIZE):
        tokens, labels = tokenize(
            tokenizer,
            task_a_texts[i:i + BATCH_SIZE],
            task_a_labels[i:i + BATCH_SIZE],
        )
        model.zero_grad()
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        F.cross_entropy(logits, labels).backward()
        if embed_layer.weight.grad is not None:
            fisher    += embed_layer.weight.grad.float() ** 2
            n_batches += 1

    fisher /= max(n_batches, 1)
    model.zero_grad()
    return fisher


# =====================================================================
# 6. DUAL TIMESCALE EMA  (for HOPE-like method)
#
# Honest dual-timescale implementation:
# - A slow EMA of the embedding tracks the post-Task-A embedding state.
# - During Task B, a penalty term pulls the fast (optimizer) weights
#   toward the slow EMA target.
# - After each optimizer step, the slow EMA is updated with decay beta,
#   allowing it to drift gradually rather than being frozen (unlike EWC).
#
# This differs meaningfully from EWC:
#   EWC:       fixed Task A snapshot, Fisher-weighted per-dimension penalty
#   HOPE-like: dynamic EMA target, uniform-weighted penalty
#
# Both use EMA_LAMBDA = EWC_LAMBDA for a fair penalty-strength comparison.
# =====================================================================
class DualTimescaleEMA:
    def __init__(self, embed_layer: nn.Embedding,
                 beta: float = EMA_BETA, lambda_penalty: float = EMA_LAMBDA):
        self.embed_layer    = embed_layer
        self.beta           = beta
        self.lambda_penalty = lambda_penalty
        # Slow EMA initialised to post-Task-A embedding
        self.slow_embed     = embed_layer.weight.detach().clone().float()

    @torch.no_grad()
    def update_slow(self):
        """EMA step — called after every Task B optimizer.step()."""
        current    = self.embed_layer.weight.float()
        self.slow_embed = self.beta * self.slow_embed + (1.0 - self.beta) * current

    def penalty(self) -> torch.Tensor:
        """L2 pull of fast weights toward slow EMA target."""
        delta = self.embed_layer.weight.float() - self.slow_embed
        return self.lambda_penalty * (delta ** 2).sum()


# =====================================================================
# 7. BASELINE — shared embedding, no protection
# =====================================================================
def train_baseline():
    print(f"\n{'=' * 60}")
    print("Method: BASELINE  (shared embedding, no protection)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)
        model.freeze_classifier_a()

        #acc_a_initial = evaluate(model, tokenizer, task_a_texts[:500], task_a_labels[:500], 'A')

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B — embedding drifts freely
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_b_step(F.cross_entropy(logits, labels), embed_layer, opt_b)

        #acc_a_final = evaluate(model, tokenizer, task_a_texts[:500], task_a_labels[:500], 'A')
        #acc_b       = evaluate(model, tokenizer, task_b_texts[:500], task_b_labels[:500], 'B')

        acc_a_final   = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b         = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')



        forgetting  = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 8. TOPOLOGICAL AI — prime embedding row anchoring
# =====================================================================
def train_topological():
    print(f"\n{'=' * 60}")
    print("Method: TOPOLOGICAL AI  (prime embedding anchors)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        # Snapshot prime rows AFTER Task A completes
        governor = TopologicalGovernor(embed_layer)
        governor.take_snapshot()
        model.freeze_classifier_a()   # explicit freeze — not covert overwriting

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:500], task_a_labels[:500], 'A')

        # Task B — restore prime rows after each step
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()              # zero prime row gradients before clip and step
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)  # same clip as all other methods
                opt_b.step()
                governor.enforce_anchors()                    # safety assertion — weights already unchanged

        assert governor.verify_integrity(), "Anchor integrity check failed!"

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b       = evaluate(model, tokenizer, task_b_texts[:500], task_b_labels[:500], 'B')
        forgetting  = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 9. EWC — empirical Fisher Information diagonal on embedding layer
# =====================================================================
def train_ewc():
    print(f"\n{'=' * 60}")
    print("Method: EWC  (Fisher Information diagonal on embedding)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        # Fisher diagonal + fixed Task A embedding snapshot
        fisher         = compute_fisher(model, tokenizer, embed_layer)
        embed_snapshot = embed_layer.weight.detach().clone().float()
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')

        # Task B with Fisher-weighted EWC penalty on embedding drift
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss  = F.cross_entropy(logits, labels)
                # EWC: Σ F_i · (θ_i − θ*_i)²
                delta      = embed_layer.weight.float() - embed_snapshot
                ewc_loss   = (fisher * delta ** 2).sum()
                task_b_step(task_loss + EWC_LAMBDA * ewc_loss, embed_layer, opt_b)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b       = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting  = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model, fisher, embed_snapshot)

    return summarise(results)


# =====================================================================
# 10. REPLAY — Task A samples correctly routed through classifier_A
#
# Key fix: Task A replay samples go through classifier_A, Task B samples
# through classifier_B. A single backward() call over the combined loss
# lets both tasks contribute gradients to the shared embedding in one pass.
#
# With classifier_A frozen (requires_grad=False), the Task A replay loss
# does not update classifier_A's weights, but gradients DO still flow
# back through classifier_A into the embedding layer — this is correct
# and intentional. The embedding receives a Task-A-preserving signal.
# =====================================================================
def train_replay():
    print(f"\n{'=' * 60}")
    print("Method: REPLAY  (Task A via classifier_A, Task B via classifier_B)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        buffer = list(zip(task_a_texts, task_a_labels))[:BUFFER_SIZE]
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                opt_b.zero_grad()

                # ── Task B forward pass ──────────────────────────────
                model.switch_task('B')
                tokens_b, labels_b = tokenize(
                    tokenizer,
                    task_b_texts[i:i + REPLAY_HALF],
                    task_b_labels[i:i + REPLAY_HALF],
                )
                logits_b, _ = model(tokens_b.input_ids, tokens_b.attention_mask)
                loss_b = F.cross_entropy(logits_b, labels_b)

                # ── Task A replay via classifier_A (correct head) ────
                replay_batch       = random.sample(buffer, min(REPLAY_HALF, len(buffer)))
                r_texts, r_labels  = zip(*replay_batch)
                tokens_a, labels_a = tokenize(tokenizer, list(r_texts), list(r_labels))
                model.switch_task('A')
                logits_a, _ = model(tokens_a.input_ids, tokens_a.attention_mask)
                loss_a = F.cross_entropy(logits_a, labels_a)

                # Combined backward: both tasks pull the shared embedding
                task_b_step(loss_b + loss_a, embed_layer, opt_b)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b       = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting  = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 11. HOPE-like — Dual Timescale EMA
# =====================================================================
def train_hope():
    print(f"\n{'=' * 60}")
    print("Method: HOPE-like  (Dual Timescale EMA on embedding)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        # EMA initialised from post-Task-A embedding
        ema = DualTimescaleEMA(embed_layer, beta=EMA_BETA, lambda_penalty=EMA_LAMBDA)
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')

        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _  = model(tokens.input_ids, tokens.attention_mask)
                task_loss  = F.cross_entropy(logits, labels)
                ema_loss   = ema.penalty()        # pulls toward slow EMA target
                task_b_step(task_loss + ema_loss, embed_layer, opt_b)
                ema.update_slow()                 # slow EMA drifts at rate (1 - beta)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b       = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting  = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 12. MAIN
# =====================================================================
if __name__ == "__main__":

    all_results = {
        'Baseline':     train_baseline(),
        'Topological':  train_topological(),
        'EWC':          train_ewc(),
        'Replay':       train_replay(),
        'HOPE-like':    train_hope(),
    }

    print("\n" + "=" * 80)
    print(f"FINAL RESULTS  (mean ± std,  N = {N_RUNS} runs per method)")
    print("=" * 80)

    col = f"{'Method':<16}  {'Forgetting':>14}  {'Task A Final':>14}  {'Task B Acc':>12}"
    print(col)
    print("-" * len(col))
    for method, res in all_results.items():
        print(
            f"{method:<16}"
            f"  {res['forgetting_mean']:>+5.1f}% ±{res['forgetting_std']:>4.1f}"
            f"  {res['acc_a_final_mean'] * 100:>6.1f}% ±{res['acc_a_final_std'] * 100:>4.1f}"
            f"  {res['acc_b_mean'] * 100:>5.1f}% ±{res['acc_b_std'] * 100:>4.1f}"
        )

    print(f"\nRANKING by forgetting (lower = better):")
    ranked = sorted(all_results.items(), key=lambda x: x[1]['forgetting_mean'])
    for i, (method, res) in enumerate(ranked, 1):
        tb = res['acc_b_mean'] * 100
        print(f"  {i}. {method:<16}  {res['forgetting_mean']:>+5.1f}%  |  Task B: {tb:.1f}%")

    print("=" * 80)

Device: cuda
CORRECTED COMPARISON: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Task A: 500 samples  (World=0, Sports=1)
Task B: 1000 samples  (Business=0, Sci/Tech=1)

Method: BASELINE  (shared embedding, no protection)

  Run 1/5


[transformers] MXFP4 quantization requires the `kernels` package: `pip install kernels>=0.12.0`. We will default to dequantizing the model to bf16.


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

...

    Task A initial: 100.0%
    Task A final:   100.0%  |  Forgetting: +0.0%
    Task B acc:     89.5%

  Run 4/5


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A initial: 100.0%
    Task A final:   100.0%  |  Forgetting: +0.0%
    Task B acc:     83.5%

  Run 5/5


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A initial: 100.0%
    Task A final:   100.0%  |  Forgetting: +0.0%
    Task B acc:     90.0%

FINAL RESULTS  (mean ± std,  N = 5 runs per method)
Method                Forgetting    Task A Final    Task B Acc
--------------------------------------------------------------
Baseline           +3.3% ± 1.4    96.5% ± 1.7   98.9% ± 0.9
Topological        +1.7% ± 0.9    98.2% ± 0.9   99.2% ± 0.5
EWC                +1.7% ± 0.8    98.1% ± 0.7   99.5% ± 0.6
Replay             -0.2% ± 0.2   100.0% ± 0.0   83.4% ± 3.6
HOPE-like          +0.0% ± 0.3    99.8% ± 0.4   87.8% ± 2.5

RANKING by forgetting (lower = better):
  1. Replay             -0.2%  |  Task B: 83.4%
  2. HOPE-like          +0.0%  |  Task B: 87.8%
  3. EWC                +1.7%  |  Task B: 99.5%
  4. Topological        +1.7%  |  Task B: 99.2%
  5. Baseline           +3.3%  |  Task B: 98.9%


## THREE-TASK BENCHMARK

In [ ]:
# =====================================================================
# AFTER EWC — FORCE GPU MEMORY FLUSH
# =====================================================================
import gc
import torch
import os

def flush_gpu_memory():
    """Force complete GPU memory release between methods."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    # Report memory state
    free, total = torch.cuda.mem_get_info()
    print(f"\n  [Memory flush] Free: {free/1024**3:.1f}GB / Total: {total/1024**3:.1f}GB")

In [ ]:
# ============================================================================
# THREE-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline
#
# Extension from two-task benchmark to three sequential tasks:
#   Task A — World vs Sports       (classes 0, 1)
#   Task B — Business vs Sci/Tech  (classes 2, 3)
#   Task C — World vs Sci/Tech     (classes 0, 3) — cross-domain, hardest
#
# Why Task C matters:
#   EWC accumulates Fisher penalties from Task A + Task B combined.
#   This growing penalty increasingly blocks Task C learning.
#   Topological hard constraints stay fixed and clean across all tasks.
#   Three tasks is where the methods are expected to diverge.
#
# Metrics reported:
#   - Task C accuracy          (learn new — primary)
#   - Forgetting A             (Task A initial - Task A after C)
#   - Forgetting B             (Task B initial - Task B after C)
#   - Combined forgetting      (mean of A and B forgetting)
#   - Protection time (ms)     (overhead of protection mechanism per task)
#   - Protection memory (KB)   (memory cost of protection mechanism)
#   - Fisher memory (MB)       (EWC only — grows with tasks)
#   - Scales to N tasks        (flat vs growing cost)
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc
import random
import time
import tracemalloc
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from warnings import filterwarnings
filterwarnings('ignore')

import sys
sys.path.insert(0, '/content/ast_lefm')
try:
    from ast_lefm.sieve import primes_up_to
except ImportError:
    def primes_up_to(n):
        """Fallback sieve of Eratosthenes."""
        sieve = [True] * (n + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(n ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, n + 1, i):
                    sieve[j] = False
        return [i for i in range(2, n + 1) if sieve[i]]


# ============================================================
# Global config
# ============================================================
SEED          = 123
N_RUNS        = 1
BATCH_SIZE    = 16
REPLAY_HALF   = 8
MAX_LEN       = 64
EPOCHS        = 6        # doubled from original
LR_EMBED      = 5e-3     # original — do not change
LR_CLS        = 1e-3     # original — do not change
EWC_LAMBDA    = 5000
EMA_BETA      = 0.9
EMA_LAMBDA    = 5000
PRIME_LIMIT   = 13       # 6 anchor rows: [2, 3, 5, 7, 11, 13]
BUFFER_SIZE   = 200
MODEL_NAME    = "openai/gpt-oss-20b"
NUM_SAMPLES_A = 500
NUM_SAMPLES_B = 1000
NUM_SAMPLES_C = 1000     # Task C — cross-domain World vs Sci/Tech
EVAL_SIZE_A   = 200
EVAL_SIZE_B   = 200
EVAL_SIZE_C   = 200


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print("=" * 80)
print("THREE-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline")
print("=" * 80)


# =====================================================================
# 1. DATASET
# =====================================================================
dataset = load_dataset("ag_news", split="train")


def create_task(dataset, class_labels, num_samples=500):
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    sampled  = filtered.select(range(min(num_samples, len(filtered))))
    texts    = [item['text']      for item in sampled]
    labels   = [item['label'] % 2 for item in sampled]
    return texts, labels


task_a_texts, task_a_labels = create_task(dataset, [0, 1], NUM_SAMPLES_A)
task_b_texts, task_b_labels = create_task(dataset, [2, 3], NUM_SAMPLES_B)
task_c_texts, task_c_labels = create_task(dataset, [0, 3], NUM_SAMPLES_C)  # World vs Sci/Tech

print(f"Task A: {len(task_a_texts)} samples  (World=0,    Sports=1)")
print(f"Task B: {len(task_b_texts)} samples  (Business=0, Sci/Tech=1)")
print(f"Task C: {len(task_c_texts)} samples  (World=0,    Sci/Tech=1)  ← cross-domain")


# =====================================================================
# 2. MODEL — SEPARATE CLASSIFIER HEAD PER TASK (now 3 heads)
# =====================================================================
class TaskAwareModel(nn.Module):
    def __init__(self, base_model, hidden_size=2880):
        super().__init__()
        self.base_model   = base_model
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs     = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        last_hidden = outputs.hidden_states[-1][:, -1, :]
        if self.current_task == 'A':
            head = self.classifier_A
        elif self.current_task == 'B':
            head = self.classifier_B
        else:
            head = self.classifier_C
        return head(last_hidden), outputs

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C'), f"Unknown task: {task}"
        self.current_task = task

    def freeze_classifier_a(self):
        self.classifier_A.requires_grad_(False)

    def freeze_classifier_b(self):
        self.classifier_B.requires_grad_(False)


# =====================================================================
# 3. SHARED UTILITIES
# =====================================================================
def get_embed_layer(base_model: nn.Module) -> nn.Embedding:
    if hasattr(base_model, 'transformer') and hasattr(base_model.transformer, 'wte'):
        return base_model.transformer.wte
    for module in base_model.modules():
        if isinstance(module, nn.Embedding) and module.weight.shape[0] > 100_000:
            return module
    raise RuntimeError("Could not locate embedding layer.")


@torch.no_grad()
def evaluate(model, tokenizer, texts, labels, task: str, batch_size: int = 32) -> float:
    was_training  = model.training
    previous_task = model.current_task
    model.eval()
    model.switch_task(task)

    all_preds, all_labels = [], []
    for i in range(0, len(texts), batch_size):
        tokens = tokenizer(
            texts[i:i + batch_size], return_tensors="pt",
            padding=True, truncation=True, max_length=MAX_LEN,
        ).to(device)
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels[i:i + batch_size])

    model.switch_task(previous_task)
    if was_training:
        model.train()
    return accuracy_score(all_labels, all_preds)


def tokenize(tokenizer, texts, labels):
    tokens = tokenizer(
        texts, return_tensors="pt",
        padding=True, truncation=True, max_length=MAX_LEN,
    ).to(device)
    return tokens, torch.tensor(labels, dtype=torch.long).to(device)


def load_fresh_model():
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True, dtype=torch.bfloat16
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = TaskAwareModel(base_model).to(device)
    for param in base_model.parameters():
        param.requires_grad = False
    return model, tokenizer, base_model


def train_task_a(model, tokenizer, optimizer):
    model.switch_task('A')
    model.train()
    for _ in range(EPOCHS):
        for i in range(0, len(task_a_texts), BATCH_SIZE):
            tokens, labels = tokenize(
                tokenizer,
                task_a_texts[i:i + BATCH_SIZE],
                task_a_labels[i:i + BATCH_SIZE],
            )
            optimizer.zero_grad()
            logits, _ = model(tokens.input_ids, tokens.attention_mask)
            F.cross_entropy(logits, labels).backward()
            optimizer.step()


def train_task_b(model, tokenizer, optimizer):
    model.switch_task('B')
    model.train()
    for _ in range(EPOCHS):
        for i in range(0, len(task_b_texts), BATCH_SIZE):
            tokens, labels = tokenize(
                tokenizer,
                task_b_texts[i:i + BATCH_SIZE],
                task_b_labels[i:i + BATCH_SIZE],
            )
            optimizer.zero_grad()
            logits, _ = model(tokens.input_ids, tokens.attention_mask)
            F.cross_entropy(logits, labels).backward()
            optimizer.step()


def build_optimizer_ab(embed_layer, model):
    """Optimizer for Task A — all heads + embedding."""
    return bnb.optim.AdamW8bit([
        {'params': embed_layer.weight,               'lr': LR_EMBED},
        {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
        {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        {'params': model.classifier_C.parameters(),  'lr': LR_CLS},
    ])


def build_task_c_optimizer(embed_layer, model):
    """Task C optimizer: embedding + classifier_C only."""
    return bnb.optim.AdamW8bit([
        {'params': embed_layer.weight,               'lr': LR_EMBED},
        {'params': model.classifier_C.parameters(),  'lr': LR_CLS},
    ])


def task_step(loss, embed_layer, optimizer, max_norm=1.0):
    loss.backward()
    torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=max_norm)
    optimizer.step()


def summarise(results: list) -> dict:
    fgt_a  = [r['forgetting_a']        for r in results]
    fgt_b  = [r['forgetting_b']        for r in results]
    fgt_c  = [r['forgetting_combined'] for r in results]
    ac     = [r['acc_c']               for r in results]
    prot_t = [r.get('protection_time_ms', 0)  for r in results]
    prot_m = [r.get('protection_mem_kb',  0)  for r in results]
    return {
        'forgetting_a_mean':        np.mean(fgt_a),  'forgetting_a_std':        np.std(fgt_a),
        'forgetting_b_mean':        np.mean(fgt_b),  'forgetting_b_std':        np.std(fgt_b),
        'forgetting_combined_mean': np.mean(fgt_c),  'forgetting_combined_std': np.std(fgt_c),
        'acc_c_mean':               np.mean(ac),     'acc_c_std':               np.std(ac),
        'protection_time_ms_mean':  np.mean(prot_t), 'protection_time_ms_std':  np.std(prot_t),
        'protection_mem_kb_mean':   np.mean(prot_m), 'protection_mem_kb_std':   np.std(prot_m),
    }


def log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c) -> tuple:
    fgt_a = (acc_a_initial - acc_a_final) * 100
    fgt_b = (acc_b_initial - acc_b_final) * 100
    fgt_combined = (fgt_a + fgt_b) / 2
    print(f"    Task A forgetting:    {fgt_a:+.1f}%  ({acc_a_initial*100:.1f}% → {acc_a_final*100:.1f}%)")
    print(f"    Task B forgetting:    {fgt_b:+.1f}%  ({acc_b_initial*100:.1f}% → {acc_b_final*100:.1f}%)")
    print(f"    Combined forgetting:  {fgt_combined:+.1f}%")
    print(f"    Task C acc:           {acc_c*100:.1f}%")
    return fgt_a, fgt_b, fgt_combined


def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


# =====================================================================
# COST TRACKING UTILITIES
# =====================================================================
def measure_time_ms(fn, *args, **kwargs):
    """Measure wall-clock time of a function call. Returns (result, ms)."""
    t0     = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, (time.perf_counter() - t0) * 1000


def fisher_memory_mb(embed_layer, n_tasks=1):
    """
    EWC stores one Fisher diagonal per task — same shape as embedding weight.
    Returns total memory in MB for n_tasks accumulated Fisher matrices (float32).
    """
    vocab_size, hidden_size = embed_layer.weight.shape
    bytes_per_matrix = vocab_size * hidden_size * 4
    return (bytes_per_matrix * n_tasks) / (1024 ** 2)


def anchor_memory_kb(governor):
    """
    Topological stores only anchor rows — fixed forever regardless of task count.
    Returns memory in KB (float32).
    """
    n_anchors   = len(governor.anchor_indices)
    hidden_size = governor.embed_layer.weight.shape[1]
    return (n_anchors * hidden_size * 4) / 1024


# =====================================================================
# 4. TOPOLOGICAL GOVERNOR
# =====================================================================
class TopologicalGovernor:
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer    = embed_layer
        vocab_size          = embed_layer.weight.shape[0]
        self.anchor_indices = [p for p in primes_up_to(prime_limit) if p < vocab_size]
        self.snapshot: dict = {}
        print(f"  → Anchoring {len(self.anchor_indices)} prime rows: {self.anchor_indices}")

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            raise RuntimeError("Call take_snapshot() before enforce_anchors().")
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )


# =====================================================================
# 5. FISHER DIAGONAL (for EWC)
#    Now supports cumulative Fisher across tasks
# =====================================================================
def compute_fisher(model, tokenizer, embed_layer, texts, labels,
                   n_samples: int = 200, task: str = 'A') -> torch.Tensor:
    model.eval()
    model.switch_task(task)
    fisher    = torch.zeros_like(embed_layer.weight, dtype=torch.float32)
    n_batches = 0

    for i in range(0, min(n_samples, len(texts)), BATCH_SIZE):
        tokens, lbs = tokenize(tokenizer, texts[i:i + BATCH_SIZE], labels[i:i + BATCH_SIZE])
        model.zero_grad()
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        F.cross_entropy(logits, lbs).backward()
        if embed_layer.weight.grad is not None:
            fisher    += embed_layer.weight.grad.float() ** 2
            n_batches += 1

    fisher /= max(n_batches, 1)
    model.zero_grad()
    return fisher


# =====================================================================
# 6. DUAL TIMESCALE EMA (for HOPE-like)
# =====================================================================
class DualTimescaleEMA:
    def __init__(self, embed_layer: nn.Embedding,
                 beta: float = EMA_BETA, lambda_penalty: float = EMA_LAMBDA):
        self.embed_layer    = embed_layer
        self.beta           = beta
        self.lambda_penalty = lambda_penalty
        self.slow_embed     = embed_layer.weight.detach().clone().float()

    @torch.no_grad()
    def update_slow(self):
        current         = self.embed_layer.weight.float()
        self.slow_embed = self.beta * self.slow_embed + (1.0 - self.beta) * current

    def penalty(self) -> torch.Tensor:
        delta = self.embed_layer.weight.float() - self.slow_embed
        return self.lambda_penalty * (delta ** 2).sum()


# =====================================================================
# 7. SHARED TASK B TRAINING (no protection — used inside each method)
# =====================================================================
def run_task_b_plain(model, tokenizer, embed_layer, optimizer):
    """Train Task B with no additional protection — used by Baseline."""
    model.switch_task('B')
    model.train()
    for _ in range(EPOCHS):
        for i in range(0, len(task_b_texts), BATCH_SIZE):
            tokens, labels = tokenize(
                tokenizer,
                task_b_texts[i:i + BATCH_SIZE],
                task_b_labels[i:i + BATCH_SIZE],
            )
            optimizer.zero_grad()
            logits, _ = model(tokens.input_ids, tokens.attention_mask)
            task_step(F.cross_entropy(logits, labels), embed_layer, optimizer)


# =====================================================================
# 8. BASELINE
# =====================================================================
def train_baseline():
    print(f"\n{'=' * 60}")
    print("Method: BASELINE  (no protection)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B — plain
        opt_b = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        run_task_b_plain(model, tokenizer, embed_layer, opt_b)
        model.freeze_classifier_b()

        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C — plain
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_step(F.cross_entropy(logits, labels), embed_layer, opt_c)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')
        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        results.append({'forgetting_a': fgt_a, 'forgetting_b': fgt_b, 'forgetting_combined': fgt_combined, 'acc_c': acc_c})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 9. TOPOLOGICAL AI
# =====================================================================
def train_topological():
    print(f"\n{'=' * 60}")
    print("Method: TOPOLOGICAL AI  (prime embedding anchors)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        # Snapshot anchors ONCE after Task A — stays fixed for ALL subsequent tasks
        governor = TopologicalGovernor(embed_layer)
        t0_snap = time.perf_counter()
        governor.take_snapshot()
        snap_time_ms = (time.perf_counter() - t0_snap) * 1000
        anchor_mem_kb = anchor_memory_kb(governor)
        print(f"    [Cost] Snapshot time: {snap_time_ms:.2f}ms | Anchor memory: {anchor_mem_kb:.2f}KB | Scales: FLAT")
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B with anchor protection
        model.switch_task('B')
        model.train()
        opt_b = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
                opt_b.step()
                governor.enforce_anchors()

        model.freeze_classifier_b()
        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C with SAME anchor protection — no new snapshot needed
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
                opt_c.step()
                governor.enforce_anchors()

        assert governor.verify_integrity(), "Anchor integrity check failed!"

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')
        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': snap_time_ms,
            'protection_mem_kb':  anchor_mem_kb,
        })
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 10. EWC — cumulative Fisher across Task A + Task B
# =====================================================================
def train_ewc():
    print(f"\n{'=' * 60}")
    print("Method: EWC  (cumulative Fisher across tasks)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        # Fisher from Task A — timed
        t0_fisher = time.perf_counter()
        fisher_a       = compute_fisher(model, tokenizer, embed_layer, task_a_texts, task_a_labels, task='A')
        embed_snap_a   = embed_layer.weight.detach().clone().float()
        fisher_time_a_ms = (time.perf_counter() - t0_fisher) * 1000
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B with EWC penalty from Task A
        model.switch_task('B')
        model.train()
        opt_b = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss = F.cross_entropy(logits, labels)
                delta_a   = embed_layer.weight.float() - embed_snap_a
                ewc_loss  = (fisher_a * delta_a ** 2).sum()
                task_step(task_loss + EWC_LAMBDA * ewc_loss, embed_layer, opt_b)

        # Fisher from Task B — cumulative with Task A Fisher — timed
        t0_fisher_b = time.perf_counter()
        fisher_b     = compute_fisher(model, tokenizer, embed_layer, task_b_texts, task_b_labels, task='B')
        embed_snap_b = embed_layer.weight.detach().clone().float()
        fisher_ab    = fisher_a + fisher_b
        fisher_time_b_ms  = (time.perf_counter() - t0_fisher_b) * 1000
        total_fisher_time = fisher_time_a_ms + fisher_time_b_ms
        fisher_mem_2tasks = fisher_memory_mb(embed_layer, n_tasks=2)
        print(f"    [Cost] Fisher A: {fisher_time_a_ms:.0f}ms | Fisher B: {fisher_time_b_ms:.0f}ms | Total: {total_fisher_time:.0f}ms | Memory (2 tasks): {fisher_mem_2tasks:.1f}MB | Scales: GROWS WITH N")
        model.freeze_classifier_b()

        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C with CUMULATIVE EWC penalty from Task A + Task B
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss  = F.cross_entropy(logits, labels)
                # Penalty from Task A snapshot
                delta_a    = embed_layer.weight.float() - embed_snap_a
                # Penalty from Task B snapshot
                delta_b    = embed_layer.weight.float() - embed_snap_b
                # Cumulative EWC: penalize drift from both previous tasks
                ewc_loss   = (fisher_ab * delta_a ** 2).sum() + (fisher_b * delta_b ** 2).sum()
                task_step(task_loss + EWC_LAMBDA * ewc_loss, embed_layer, opt_c)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')
        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': total_fisher_time,
            'protection_mem_kb':  fisher_mem_2tasks * 1024,
        })
        cleanup(model, base_model, fisher_a, fisher_b, fisher_ab, embed_snap_a, embed_snap_b)

    return summarise(results)


# =====================================================================
# 11. REPLAY
# =====================================================================
def train_replay():
    print(f"\n{'=' * 60}")
    print("Method: REPLAY  (buffer from Task A and Task B)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        buffer_a = list(zip(task_a_texts, task_a_labels))[:BUFFER_SIZE]
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        model.train()

        # Task B with Task A replay
        opt_b = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                opt_b.zero_grad()
                model.switch_task('B')
                tokens_b, labels_b = tokenize(tokenizer, task_b_texts[i:i + REPLAY_HALF], task_b_labels[i:i + REPLAY_HALF])
                logits_b, _ = model(tokens_b.input_ids, tokens_b.attention_mask)
                loss_b = F.cross_entropy(logits_b, labels_b)

                replay_a      = random.sample(buffer_a, min(REPLAY_HALF, len(buffer_a)))
                r_texts, r_lbs = zip(*replay_a)
                tokens_a, labels_a = tokenize(tokenizer, list(r_texts), list(r_lbs))
                model.switch_task('A')
                logits_a, _ = model(tokens_a.input_ids, tokens_a.attention_mask)
                loss_a = F.cross_entropy(logits_a, labels_a)
                task_step(loss_b + loss_a, embed_layer, opt_b)

        buffer_b = list(zip(task_b_texts, task_b_labels))[:BUFFER_SIZE]
        model.freeze_classifier_b()
        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C with Task A + Task B replay
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                opt_c.zero_grad()
                model.switch_task('C')
                tokens_c, labels_c = tokenize(tokenizer, task_c_texts[i:i + REPLAY_HALF], task_c_labels[i:i + REPLAY_HALF])
                logits_c, _ = model(tokens_c.input_ids, tokens_c.attention_mask)
                loss_c = F.cross_entropy(logits_c, labels_c)

                # Replay from Task A
                replay_a        = random.sample(buffer_a, min(REPLAY_HALF // 2, len(buffer_a)))
                r_texts, r_lbs  = zip(*replay_a)
                tokens_a, lbs_a = tokenize(tokenizer, list(r_texts), list(r_lbs))
                model.switch_task('A')
                logits_a, _ = model(tokens_a.input_ids, tokens_a.attention_mask)
                loss_a = F.cross_entropy(logits_a, lbs_a)

                # Replay from Task B
                replay_b        = random.sample(buffer_b, min(REPLAY_HALF // 2, len(buffer_b)))
                r_texts, r_lbs  = zip(*replay_b)
                tokens_b, lbs_b = tokenize(tokenizer, list(r_texts), list(r_lbs))
                model.switch_task('B')
                logits_b, _ = model(tokens_b.input_ids, tokens_b.attention_mask)
                loss_b = F.cross_entropy(logits_b, lbs_b)

                task_step(loss_c + loss_a + loss_b, embed_layer, opt_c)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')
        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        results.append({'forgetting_a': fgt_a, 'forgetting_b': fgt_b, 'forgetting_combined': fgt_combined, 'acc_c': acc_c})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 12. HOPE-like
# =====================================================================
def train_hope():
    print(f"\n{'=' * 60}")
    print("Method: HOPE-like  (Dual Timescale EMA)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        ema = DualTimescaleEMA(embed_layer, beta=EMA_BETA, lambda_penalty=EMA_LAMBDA)
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B with EMA penalty
        model.switch_task('B')
        model.train()
        opt_b = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _  = model(tokens.input_ids, tokens.attention_mask)
                task_loss  = F.cross_entropy(logits, labels)
                task_step(task_loss + ema.penalty(), embed_layer, opt_b)
                ema.update_slow()

        # Update EMA target after Task B — now tracks post-B state
        model.freeze_classifier_b()
        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C — EMA now pulls toward post-B embedding
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _  = model(tokens.input_ids, tokens.attention_mask)
                task_loss  = F.cross_entropy(logits, labels)
                task_step(task_loss + ema.penalty(), embed_layer, opt_c)
                ema.update_slow()

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')
        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        results.append({'forgetting_a': fgt_a, 'forgetting_b': fgt_b, 'forgetting_combined': fgt_combined, 'acc_c': acc_c})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 13. MAIN
# =====================================================================
if __name__ == "__main__":

    def flush_gpu_memory():
        """Force complete GPU memory release between methods."""
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        free, total = torch.cuda.mem_get_info()
        print(f"\n  [Memory flush] Free: {free/1024**3:.1f}GB / Total: {total/1024**3:.1f}GB")

    all_results = {}

    all_results['Baseline']    = train_baseline();    flush_gpu_memory()
    all_results['Topological'] = train_topological(); flush_gpu_memory()
    all_results['EWC']         = train_ewc();         flush_gpu_memory()  # critical — releases 4.4GB Fisher
    all_results['Replay']      = train_replay();      flush_gpu_memory()
    all_results['HOPE-like']   = train_hope()

    print("\n" + "=" * 80)
    print(f"FINAL RESULTS — THREE TASKS  (mean ± std,  N = {N_RUNS} runs)")
    print("=" * 80)

    col = f"{'Method':<16}  {'Task C Acc':>12}  {'Fgt A':>8}  {'Fgt B':>8}  {'Fgt Comb':>10}  {'Prot Time':>12}  {'Prot Mem':>12}"
    print(col)
    print("-" * len(col))
    for method, res in all_results.items():
        print(
            f"{method:<16}"
            f"  {res['acc_c_mean']*100:>5.1f}% ±{res['acc_c_std']*100:>4.1f}"
            f"  {res['forgetting_a_mean']:>+5.1f}%"
            f"  {res['forgetting_b_mean']:>+5.1f}%"
            f"  {res['forgetting_combined_mean']:>+6.1f}%"
            f"  {res['protection_time_ms_mean']:>7.0f}ms"
            f"  {res['protection_mem_kb_mean']:>7.1f}KB"
        )

    print(f"\nRANKING by Task C accuracy (higher = better):")
    ranked = sorted(all_results.items(), key=lambda x: x[1]['acc_c_mean'], reverse=True)
    for i, (method, res) in enumerate(ranked, 1):
        print(
            f"  {i}. {method:<16}"
            f"  Task C: {res['acc_c_mean']*100:.1f}%"
            f"  |  Combined forgetting: {res['forgetting_combined_mean']:+.1f}%"
            f"  |  Protection: {res['protection_time_ms_mean']:.0f}ms / {res['protection_mem_kb_mean']:.1f}KB"
        )

    print(f"\nCOST SCALING NOTE:")
    print(f"  Topological AI — protection cost is FLAT regardless of number of tasks")
    print(f"  EWC             — Fisher memory grows linearly: ~{all_results['EWC']['protection_mem_kb_mean']/1024:.0f}MB per task added")
    print(f"  EWC at 10 tasks — estimated memory: ~{all_results['EWC']['protection_mem_kb_mean']/1024*10/1024:.1f}GB")
    print("=" * 80)

Device: cuda
THREE-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline
Task A: 500 samples  (World=0,    Sports=1)
Task B: 1000 samples  (Business=0, Sci/Tech=1)
Task C: 1000 samples  (World=0,    Sci/Tech=1)  ← cross-domain

Method: BASELINE  (no protection)

  Run 1/1


[transformers] MXFP4 quantization requires the `kernels` package: `pip install kernels>=0.12.0`. We will default to dequantizing the model to bf16.


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A forgetting:    +6.5%  (99.5% → 93.0%)
    Task B forgetting:    +3.5%  (98.5% → 95.0%)
    Combined forgetting:  +5.0%
    Task C acc:           97.5%

  [Memory flush] Free: 94.2GB / Total: 95.0GB

Method: TOPOLOGICAL AI  (prime embedding anchors)

  Run 1/1


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

...

    [Cost] Fisher A: 2419ms | Fisher B: 2427ms | Total: 4846ms | Memory (2 tasks): 4418.4MB | Scales: GROWS WITH N
    Task A forgetting:    +6.5%  (99.5% → 93.0%)
    Task B forgetting:    +7.0%  (99.0% → 92.0%)
    Combined forgetting:  +6.7%
    Task C acc:           98.5%

  [Memory flush] Free: 94.2GB / Total: 95.0GB

Method: REPLAY  (buffer from Task A and Task B)

  Run 1/1


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A forgetting:    -0.5%  (99.5% → 100.0%)
    Task B forgetting:    -20.0%  (78.5% → 98.5%)
    Combined forgetting:  -10.2%
    Task C acc:           94.0%

  [Memory flush] Free: 94.2GB / Total: 95.0GB

Method: HOPE-like  (Dual Timescale EMA)

  Run 1/1


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A forgetting:    -0.5%  (99.5% → 100.0%)
    Task B forgetting:    +0.5%  (89.5% → 89.0%)
    Combined forgetting:  +0.0%
    Task C acc:           82.0%

FINAL RESULTS — THREE TASKS  (mean ± std,  N = 1 runs)
Method              Task C Acc     Fgt A     Fgt B    Fgt Comb     Prot Time      Prot Mem
------------------------------------------------------------------------------------------
Baseline           97.5% ± 0.0   +6.5%   +3.5%    +5.0%        0ms      0.0KB
Topological        99.5% ± 0.0   +6.0%   +2.5%    +4.2%        0ms     67.5KB
EWC                98.5% ± 0.0   +6.5%   +7.0%    +6.7%     4846ms  4524480.0KB
Replay             94.0% ± 0.0   -0.5%  -20.0%   -10.2%        0ms      0.0KB
HOPE-like          82.0% ± 0.0   -0.5%   +0.5%    +0.0%        0ms      0.0KB

RANKING by Task C accuracy (higher = better):
  1. Topological       Task C: 99.5%  |  Combined forgetting: +4.2%  |  Protection: 0ms / 67.5KB
  2. EWC               Task C: 98.5%  |  Combined forgetting: +

## THREE-TASK BENCHMARK - NEWVERSION

In [ ]:
# ============================================================================
# THREE-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline
#
# Extension from two-task benchmark to three sequential tasks:
#   Task A — World vs Sports       (classes 0, 1)
#   Task B — Business vs Sci/Tech  (classes 2, 3)
#   Task C — World vs Sci/Tech     (classes 0, 3) — cross-domain, hardest
#
# Why Task C matters:
#   EWC accumulates Fisher penalties from Task A + Task B combined.
#   This growing penalty increasingly blocks Task C learning.
#   Topological hard constraints stay fixed and clean across all tasks.
#   Three tasks is where the methods are expected to diverge.
#
# Metrics reported:
#   - Task C accuracy          (learn new — primary)
#   - Forgetting A             (Task A initial - Task A after C)
#   - Forgetting B             (Task B initial - Task B after C)
#   - Combined forgetting      (mean of A and B forgetting)
#   - Protection time (ms)     (overhead of protection mechanism per task)
#   - Protection memory (KB)   (memory cost of protection mechanism)
#   - Fisher memory (MB)       (EWC only — grows with tasks)
#   - Scales to N tasks        (flat vs growing cost)
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc
import random
import time
import tracemalloc
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from warnings import filterwarnings
filterwarnings('ignore')

import sys
sys.path.insert(0, '/content/ast_lefm')
try:
    from ast_lefm.sieve import primes_up_to
except ImportError:
    def primes_up_to(n):
        """Fallback sieve of Eratosthenes."""
        sieve = [True] * (n + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(n ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, n + 1, i):
                    sieve[j] = False
        return [i for i in range(2, n + 1) if sieve[i]]


# ============================================================
# Global config
# ============================================================
SEED          = 123
N_RUNS        = 1
BATCH_SIZE    = 16
REPLAY_HALF   = 8
MAX_LEN       = 64
EPOCHS        = 6        # doubled from original
LR_EMBED      = 5e-3     # original — do not change
LR_CLS        = 1e-3     # original — do not change
EWC_LAMBDA    = 5000
EMA_BETA      = 0.9
EMA_LAMBDA    = 5000
PRIME_LIMIT   = 13       # 6 anchor rows: [2, 3, 5, 7, 11, 13]
BUFFER_SIZE   = 200
MODEL_NAME    = "openai/gpt-oss-20b"
NUM_SAMPLES_A = 500
NUM_SAMPLES_B = 1000
NUM_SAMPLES_C = 1000     # Task C — cross-domain World vs Sci/Tech
EVAL_SIZE_A   = 200
EVAL_SIZE_B   = 200
EVAL_SIZE_C   = 200


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print("=" * 80)
print("THREE-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline")
print("=" * 80)


# =====================================================================
# 1. DATASET
# =====================================================================
dataset = load_dataset("ag_news", split="train")


def create_task(dataset, class_labels, num_samples=500):
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    sampled  = filtered.select(range(min(num_samples, len(filtered))))
    texts    = [item['text']      for item in sampled]
    labels   = [item['label'] % 2 for item in sampled]
    return texts, labels


task_a_texts, task_a_labels = create_task(dataset, [0, 1], NUM_SAMPLES_A)
task_b_texts, task_b_labels = create_task(dataset, [2, 3], NUM_SAMPLES_B)
task_c_texts, task_c_labels = create_task(dataset, [0, 3], NUM_SAMPLES_C)  # World vs Sci/Tech

print(f"Task A: {len(task_a_texts)} samples  (World=0,    Sports=1)")
print(f"Task B: {len(task_b_texts)} samples  (Business=0, Sci/Tech=1)")
print(f"Task C: {len(task_c_texts)} samples  (World=0,    Sci/Tech=1)  ← cross-domain")


# =====================================================================
# 2. MODEL — SEPARATE CLASSIFIER HEAD PER TASK (now 3 heads)
# =====================================================================
class TaskAwareModel(nn.Module):
    def __init__(self, base_model, hidden_size=2880):
        super().__init__()
        self.base_model   = base_model
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs     = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        last_hidden = outputs.hidden_states[-1][:, -1, :]
        if self.current_task == 'A':
            head = self.classifier_A
        elif self.current_task == 'B':
            head = self.classifier_B
        else:
            head = self.classifier_C
        return head(last_hidden), outputs

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C'), f"Unknown task: {task}"
        self.current_task = task

    def freeze_classifier_a(self):
        self.classifier_A.requires_grad_(False)

    def freeze_classifier_b(self):
        self.classifier_B.requires_grad_(False)


# =====================================================================
# 3. SHARED UTILITIES
# =====================================================================
def get_embed_layer(base_model: nn.Module) -> nn.Embedding:
    if hasattr(base_model, 'transformer') and hasattr(base_model.transformer, 'wte'):
        return base_model.transformer.wte
    for module in base_model.modules():
        if isinstance(module, nn.Embedding) and module.weight.shape[0] > 100_000:
            return module
    raise RuntimeError("Could not locate embedding layer.")


@torch.no_grad()
def evaluate(model, tokenizer, texts, labels, task: str, batch_size: int = 32) -> float:
    was_training  = model.training
    previous_task = model.current_task
    model.eval()
    model.switch_task(task)

    all_preds, all_labels = [], []
    for i in range(0, len(texts), batch_size):
        tokens = tokenizer(
            texts[i:i + batch_size], return_tensors="pt",
            padding=True, truncation=True, max_length=MAX_LEN,
        ).to(device)
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels[i:i + batch_size])

    model.switch_task(previous_task)
    if was_training:
        model.train()
    return accuracy_score(all_labels, all_preds)


def tokenize(tokenizer, texts, labels):
    tokens = tokenizer(
        texts, return_tensors="pt",
        padding=True, truncation=True, max_length=MAX_LEN,
    ).to(device)
    return tokens, torch.tensor(labels, dtype=torch.long).to(device)


def load_fresh_model():
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True, dtype=torch.bfloat16
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = TaskAwareModel(base_model).to(device)
    for param in base_model.parameters():
        param.requires_grad = False
    return model, tokenizer, base_model


def train_task_a(model, tokenizer, optimizer):
    model.switch_task('A')
    model.train()
    for _ in range(EPOCHS):
        for i in range(0, len(task_a_texts), BATCH_SIZE):
            tokens, labels = tokenize(
                tokenizer,
                task_a_texts[i:i + BATCH_SIZE],
                task_a_labels[i:i + BATCH_SIZE],
            )
            optimizer.zero_grad()
            logits, _ = model(tokens.input_ids, tokens.attention_mask)
            F.cross_entropy(logits, labels).backward()
            optimizer.step()


def train_task_b(model, tokenizer, optimizer):
    model.switch_task('B')
    model.train()
    for _ in range(EPOCHS):
        for i in range(0, len(task_b_texts), BATCH_SIZE):
            tokens, labels = tokenize(
                tokenizer,
                task_b_texts[i:i + BATCH_SIZE],
                task_b_labels[i:i + BATCH_SIZE],
            )
            optimizer.zero_grad()
            logits, _ = model(tokens.input_ids, tokens.attention_mask)
            F.cross_entropy(logits, labels).backward()
            optimizer.step()


def build_optimizer_ab(embed_layer, model):
    """Optimizer for Task A — all heads + embedding."""
    return bnb.optim.AdamW8bit([
        {'params': embed_layer.weight,               'lr': LR_EMBED},
        {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
        {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        {'params': model.classifier_C.parameters(),  'lr': LR_CLS},
    ])


def build_task_c_optimizer(embed_layer, model):
    """Task C optimizer: embedding + classifier_C only."""
    return bnb.optim.AdamW8bit([
        {'params': embed_layer.weight,               'lr': LR_EMBED},
        {'params': model.classifier_C.parameters(),  'lr': LR_CLS},
    ])


def task_step(loss, embed_layer, optimizer, max_norm=1.0):
    loss.backward()
    torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=max_norm)
    optimizer.step()


def summarise(results: list) -> dict:
    fgt_a  = [r['forgetting_a']        for r in results]
    fgt_b  = [r['forgetting_b']        for r in results]
    fgt_c  = [r['forgetting_combined'] for r in results]
    ac     = [r['acc_c']               for r in results]
    prot_t = [r.get('protection_time_ms', 0)  for r in results]
    prot_m = [r.get('protection_mem_kb',  0)  for r in results]
    return {
        'forgetting_a_mean':        np.mean(fgt_a),  'forgetting_a_std':        np.std(fgt_a),
        'forgetting_b_mean':        np.mean(fgt_b),  'forgetting_b_std':        np.std(fgt_b),
        'forgetting_combined_mean': np.mean(fgt_c),  'forgetting_combined_std': np.std(fgt_c),
        'acc_c_mean':               np.mean(ac),     'acc_c_std':               np.std(ac),
        'protection_time_ms_mean':  np.mean(prot_t), 'protection_time_ms_std':  np.std(prot_t),
        'protection_mem_kb_mean':   np.mean(prot_m), 'protection_mem_kb_std':   np.std(prot_m),
    }


def log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c) -> tuple:
    fgt_a = (acc_a_initial - acc_a_final) * 100
    fgt_b = (acc_b_initial - acc_b_final) * 100
    fgt_combined = (fgt_a + fgt_b) / 2
    print(f"    Task A forgetting:    {fgt_a:+.1f}%  ({acc_a_initial*100:.1f}% → {acc_a_final*100:.1f}%)")
    print(f"    Task B forgetting:    {fgt_b:+.1f}%  ({acc_b_initial*100:.1f}% → {acc_b_final*100:.1f}%)")
    print(f"    Combined forgetting:  {fgt_combined:+.1f}%")
    print(f"    Task C acc:           {acc_c*100:.1f}%")
    return fgt_a, fgt_b, fgt_combined


def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


# =====================================================================
# COST TRACKING UTILITIES
# =====================================================================
def measure_time_ms(fn, *args, **kwargs):
    """Measure wall-clock time of a function call. Returns (result, ms)."""
    t0     = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, (time.perf_counter() - t0) * 1000


def fisher_memory_mb(embed_layer, n_tasks=1):
    """
    EWC stores one Fisher diagonal per task — same shape as embedding weight.
    Returns total memory in MB for n_tasks accumulated Fisher matrices (float32).
    """
    vocab_size, hidden_size = embed_layer.weight.shape
    bytes_per_matrix = vocab_size * hidden_size * 4
    return (bytes_per_matrix * n_tasks) / (1024 ** 2)


def anchor_memory_kb(governor):
    """
    Topological stores only anchor rows — fixed forever regardless of task count.
    Returns memory in KB (float32).
    """
    n_anchors   = len(governor.anchor_indices)
    hidden_size = governor.embed_layer.weight.shape[1]
    return (n_anchors * hidden_size * 4) / 1024


# =====================================================================
# 4. TOPOLOGICAL GOVERNOR
# =====================================================================
class TopologicalGovernor:
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer    = embed_layer
        vocab_size          = embed_layer.weight.shape[0]
        self.anchor_indices = [p for p in primes_up_to(prime_limit) if p < vocab_size]
        self.snapshot: dict = {}
        print(f"  → Anchoring {len(self.anchor_indices)} prime rows: {self.anchor_indices}")

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            raise RuntimeError("Call take_snapshot() before enforce_anchors().")
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )


# =====================================================================
# 5. FISHER DIAGONAL (for EWC)
#    Now supports cumulative Fisher across tasks
# =====================================================================
def compute_fisher(model, tokenizer, embed_layer, texts, labels,
                   n_samples: int = 200, task: str = 'A') -> torch.Tensor:
    model.eval()
    model.switch_task(task)
    fisher    = torch.zeros_like(embed_layer.weight, dtype=torch.float32)
    n_batches = 0

    for i in range(0, min(n_samples, len(texts)), BATCH_SIZE):
        tokens, lbs = tokenize(tokenizer, texts[i:i + BATCH_SIZE], labels[i:i + BATCH_SIZE])
        model.zero_grad()
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        F.cross_entropy(logits, lbs).backward()
        if embed_layer.weight.grad is not None:
            fisher    += embed_layer.weight.grad.float() ** 2
            n_batches += 1

    fisher /= max(n_batches, 1)
    model.zero_grad()
    return fisher


# =====================================================================
# 6. DUAL TIMESCALE EMA (for HOPE-like)
# =====================================================================
class DualTimescaleEMA:
    def __init__(self, embed_layer: nn.Embedding,
                 beta: float = EMA_BETA, lambda_penalty: float = EMA_LAMBDA):
        self.embed_layer    = embed_layer
        self.beta           = beta
        self.lambda_penalty = lambda_penalty
        self.slow_embed     = embed_layer.weight.detach().clone().float()

    @torch.no_grad()
    def update_slow(self):
        current         = self.embed_layer.weight.float()
        self.slow_embed = self.beta * self.slow_embed + (1.0 - self.beta) * current

    def penalty(self) -> torch.Tensor:
        delta = self.embed_layer.weight.float() - self.slow_embed
        return self.lambda_penalty * (delta ** 2).sum()


# =====================================================================
# 7. SHARED TASK B TRAINING (no protection — used inside each method)
# =====================================================================
def run_task_b_plain(model, tokenizer, embed_layer, optimizer):
    """Train Task B with no additional protection — used by Baseline."""
    model.switch_task('B')
    model.train()
    for _ in range(EPOCHS):
        for i in range(0, len(task_b_texts), BATCH_SIZE):
            tokens, labels = tokenize(
                tokenizer,
                task_b_texts[i:i + BATCH_SIZE],
                task_b_labels[i:i + BATCH_SIZE],
            )
            optimizer.zero_grad()
            logits, _ = model(tokens.input_ids, tokens.attention_mask)
            task_step(F.cross_entropy(logits, labels), embed_layer, optimizer)


# =====================================================================
# 8. BASELINE
# =====================================================================
def train_baseline():
    print(f"\n{'=' * 60}")
    print("Method: BASELINE  (no protection)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B — plain
        opt_b = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        run_task_b_plain(model, tokenizer, embed_layer, opt_b)
        model.freeze_classifier_b()

        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C — plain
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_step(F.cross_entropy(logits, labels), embed_layer, opt_c)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')
        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        print(f"    [Cost] No protection mechanism | Time: 0ms | Memory: 0KB")
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': 0.0,
            'protection_mem_kb':  0.0,
        })
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 9. TOPOLOGICAL AI
# =====================================================================
def train_topological():
    print(f"\n{'=' * 60}")
    print("Method: TOPOLOGICAL AI  (prime embedding anchors)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        # Snapshot anchors ONCE after Task A — stays fixed for ALL subsequent tasks
        governor = TopologicalGovernor(embed_layer)
        t0_snap = time.perf_counter()
        governor.take_snapshot()
        snap_time_ms = (time.perf_counter() - t0_snap) * 1000
        anchor_mem_kb = anchor_memory_kb(governor)
        print(f"    [Cost] Snapshot time: {snap_time_ms:.2f}ms | Anchor memory: {anchor_mem_kb:.2f}KB | Scales: FLAT")
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B with anchor protection
        model.switch_task('B')
        model.train()
        opt_b = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
                opt_b.step()
                governor.enforce_anchors()

        model.freeze_classifier_b()
        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C with SAME anchor protection — no new snapshot needed
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
                opt_c.step()
                governor.enforce_anchors()

        assert governor.verify_integrity(), "Anchor integrity check failed!"

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')
        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': snap_time_ms,
            'protection_mem_kb':  anchor_mem_kb,
        })
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 10. EWC — cumulative Fisher across Task A + Task B
# =====================================================================
def train_ewc():
    print(f"\n{'=' * 60}")
    print("Method: EWC  (cumulative Fisher across tasks)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        # Fisher from Task A — timed
        t0_fisher = time.perf_counter()
        fisher_a       = compute_fisher(model, tokenizer, embed_layer, task_a_texts, task_a_labels, task='A')
        embed_snap_a   = embed_layer.weight.detach().clone().float()
        fisher_time_a_ms = (time.perf_counter() - t0_fisher) * 1000
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B with EWC penalty from Task A
        model.switch_task('B')
        model.train()
        opt_b = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss = F.cross_entropy(logits, labels)
                delta_a   = embed_layer.weight.float() - embed_snap_a
                ewc_loss  = (fisher_a * delta_a ** 2).sum()
                task_step(task_loss + EWC_LAMBDA * ewc_loss, embed_layer, opt_b)

        # Fisher from Task B — cumulative with Task A Fisher — timed
        t0_fisher_b = time.perf_counter()
        fisher_b     = compute_fisher(model, tokenizer, embed_layer, task_b_texts, task_b_labels, task='B')
        embed_snap_b = embed_layer.weight.detach().clone().float()
        fisher_ab    = fisher_a + fisher_b
        fisher_time_b_ms  = (time.perf_counter() - t0_fisher_b) * 1000
        total_fisher_time = fisher_time_a_ms + fisher_time_b_ms
        fisher_mem_2tasks = fisher_memory_mb(embed_layer, n_tasks=2)
        print(f"    [Cost] Fisher A: {fisher_time_a_ms:.0f}ms | Fisher B: {fisher_time_b_ms:.0f}ms | Total: {total_fisher_time:.0f}ms | Memory (2 tasks): {fisher_mem_2tasks:.1f}MB | Scales: GROWS WITH N")
        model.freeze_classifier_b()

        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C with CUMULATIVE EWC penalty from Task A + Task B
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss  = F.cross_entropy(logits, labels)
                # Penalty from Task A snapshot
                delta_a    = embed_layer.weight.float() - embed_snap_a
                # Penalty from Task B snapshot
                delta_b    = embed_layer.weight.float() - embed_snap_b
                # Cumulative EWC: penalize drift from both previous tasks
                ewc_loss   = (fisher_ab * delta_a ** 2).sum() + (fisher_b * delta_b ** 2).sum()
                task_step(task_loss + EWC_LAMBDA * ewc_loss, embed_layer, opt_c)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')
        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': total_fisher_time,
            'protection_mem_kb':  fisher_mem_2tasks * 1024,
        })
        cleanup(model, base_model, fisher_a, fisher_b, fisher_ab, embed_snap_a, embed_snap_b)

    return summarise(results)


# =====================================================================
# 11. REPLAY
# =====================================================================
def train_replay():
    print(f"\n{'=' * 60}")
    print("Method: REPLAY  (buffer from Task A and Task B)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        # Measure buffer memory — two buffers (A and B) of BUFFER_SIZE samples each
        buffer_a = list(zip(task_a_texts, task_a_labels))[:BUFFER_SIZE]
        t0_replay = time.perf_counter()
        buffer_b_size_kb = (BUFFER_SIZE * 2 * MAX_LEN * 4) / 1024  # approx: tokens + labels, float32
        buffer_a_size_kb = buffer_b_size_kb
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        model.train()

        # Task B with Task A replay
        opt_b = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                opt_b.zero_grad()
                model.switch_task('B')
                tokens_b, labels_b = tokenize(tokenizer, task_b_texts[i:i + REPLAY_HALF], task_b_labels[i:i + REPLAY_HALF])
                logits_b, _ = model(tokens_b.input_ids, tokens_b.attention_mask)
                loss_b = F.cross_entropy(logits_b, labels_b)

                replay_a      = random.sample(buffer_a, min(REPLAY_HALF, len(buffer_a)))
                r_texts, r_lbs = zip(*replay_a)
                tokens_a, labels_a = tokenize(tokenizer, list(r_texts), list(r_lbs))
                model.switch_task('A')
                logits_a, _ = model(tokens_a.input_ids, tokens_a.attention_mask)
                loss_a = F.cross_entropy(logits_a, labels_a)
                task_step(loss_b + loss_a, embed_layer, opt_b)

        buffer_b = list(zip(task_b_texts, task_b_labels))[:BUFFER_SIZE]
        model.freeze_classifier_b()
        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C with Task A + Task B replay
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                opt_c.zero_grad()
                model.switch_task('C')
                tokens_c, labels_c = tokenize(tokenizer, task_c_texts[i:i + REPLAY_HALF], task_c_labels[i:i + REPLAY_HALF])
                logits_c, _ = model(tokens_c.input_ids, tokens_c.attention_mask)
                loss_c = F.cross_entropy(logits_c, labels_c)

                # Replay from Task A
                replay_a        = random.sample(buffer_a, min(REPLAY_HALF // 2, len(buffer_a)))
                r_texts, r_lbs  = zip(*replay_a)
                tokens_a, lbs_a = tokenize(tokenizer, list(r_texts), list(r_lbs))
                model.switch_task('A')
                logits_a, _ = model(tokens_a.input_ids, tokens_a.attention_mask)
                loss_a = F.cross_entropy(logits_a, lbs_a)

                # Replay from Task B
                replay_b        = random.sample(buffer_b, min(REPLAY_HALF // 2, len(buffer_b)))
                r_texts, r_lbs  = zip(*replay_b)
                tokens_b, lbs_b = tokenize(tokenizer, list(r_texts), list(r_lbs))
                model.switch_task('B')
                logits_b, _ = model(tokens_b.input_ids, tokens_b.attention_mask)
                loss_b = F.cross_entropy(logits_b, lbs_b)

                task_step(loss_c + loss_a + loss_b, embed_layer, opt_c)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')
        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        # Buffer memory: two buffers stored in RAM
        total_buffer_kb = (len(buffer_a) + len(buffer_b)) * MAX_LEN * 4 / 1024
        # Replay overhead time: timer started at buffer_a creation
        replay_time_ms = (time.perf_counter() - t0_replay) * 1000
        print(f"    [Cost] Buffer memory: {total_buffer_kb:.1f}KB (A+B) | Total replay time: {replay_time_ms:.0f}ms | Scales: GROWS WITH N")
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': replay_time_ms,
            'protection_mem_kb':  total_buffer_kb,
        })
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 12. HOPE-like
# =====================================================================
def train_hope():
    print(f"\n{'=' * 60}")
    print("Method: HOPE-like  (Dual Timescale EMA)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        t0_ema = time.perf_counter()
        ema = DualTimescaleEMA(embed_layer, beta=EMA_BETA, lambda_penalty=EMA_LAMBDA)
        # EMA slow_embed memory: same shape as embedding weight (float32)
        vocab_size, hidden_size = embed_layer.weight.shape
        ema_mem_kb = (vocab_size * hidden_size * 4) / 1024
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B with EMA penalty
        model.switch_task('B')
        model.train()
        opt_b = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight,               'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        ])
        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _  = model(tokens.input_ids, tokens.attention_mask)
                task_loss  = F.cross_entropy(logits, labels)
                task_step(task_loss + ema.penalty(), embed_layer, opt_b)
                ema.update_slow()

        # Update EMA target after Task B — now tracks post-B state
        model.freeze_classifier_b()
        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C — EMA now pulls toward post-B embedding
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _  = model(tokens.input_ids, tokens.attention_mask)
                task_loss  = F.cross_entropy(logits, labels)
                task_step(task_loss + ema.penalty(), embed_layer, opt_c)
                ema.update_slow()

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')
        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        ema_time_ms = (time.perf_counter() - t0_ema) * 1000
        print(f"    [Cost] EMA memory: {ema_mem_kb:.1f}KB (slow embed) | Total EMA time: {ema_time_ms:.0f}ms | Scales: FLAT")
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': ema_time_ms,
            'protection_mem_kb':  ema_mem_kb,
        })
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 13. MAIN
# =====================================================================
if __name__ == "__main__":

    def flush_gpu_memory():
        """Force complete GPU memory release between methods."""
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        free, total = torch.cuda.mem_get_info()
        print(f"\n  [Memory flush] Free: {free/1024**3:.1f}GB / Total: {total/1024**3:.1f}GB")

    all_results = {}
    all_results['Baseline']    = train_baseline();    flush_gpu_memory()
    all_results['Topological'] = train_topological(); flush_gpu_memory()
    all_results['EWC']         = train_ewc();         flush_gpu_memory()
    all_results['Replay']      = train_replay();      flush_gpu_memory()
    all_results['HOPE-like']   = train_hope()

    print("\n" + "=" * 80)
    print(f"FINAL RESULTS — THREE TASKS  (mean ± std,  N = {N_RUNS} runs)")
    print("=" * 80)

    col = f"{'Method':<16}  {'Task C Acc':>12}  {'Fgt A':>8}  {'Fgt B':>8}  {'Fgt Comb':>10}  {'Prot Time':>12}  {'Prot Mem':>12}"
    print(col)
    print("-" * len(col))
    for method, res in all_results.items():
        print(
            f"{method:<16}"
            f"  {res['acc_c_mean']*100:>5.1f}% ±{res['acc_c_std']*100:>4.1f}"
            f"  {res['forgetting_a_mean']:>+5.1f}%"
            f"  {res['forgetting_b_mean']:>+5.1f}%"
            f"  {res['forgetting_combined_mean']:>+6.1f}%"
            f"  {res['protection_time_ms_mean']:>7.0f}ms"
            f"  {res['protection_mem_kb_mean']:>7.1f}KB"
        )

    print(f"\nRANKING by Task C accuracy (higher = better):")
    ranked = sorted(all_results.items(), key=lambda x: x[1]['acc_c_mean'], reverse=True)
    for i, (method, res) in enumerate(ranked, 1):
        print(
            f"  {i}. {method:<16}"
            f"  Task C: {res['acc_c_mean']*100:.1f}%"
            f"  |  Combined forgetting: {res['forgetting_combined_mean']:+.1f}%"
            f"  |  Protection: {res['protection_time_ms_mean']:.0f}ms / {res['protection_mem_kb_mean']:.1f}KB"
        )

    print(f"\nCOST SCALING NOTE:")
    print(f"  Topological AI — protection cost is FLAT regardless of number of tasks")
    print(f"  EWC             — Fisher memory grows linearly: ~{all_results['EWC']['protection_mem_kb_mean']/1024:.0f}MB per task added")
    print("=" * 80)

Device: cuda
THREE-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline
Task A: 500 samples  (World=0,    Sports=1)
Task B: 1000 samples  (Business=0, Sci/Tech=1)
Task C: 1000 samples  (World=0,    Sci/Tech=1)  ← cross-domain

Method: BASELINE  (no protection)

  Run 1/1


[transformers] MXFP4 quantization requires the `kernels` package: `pip install kernels>=0.12.0`. We will default to dequantizing the model to bf16.


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A forgetting:    +6.5%  (99.5% → 93.0%)
    Task B forgetting:    +3.5%  (98.5% → 95.0%)
    Combined forgetting:  +5.0%
    Task C acc:           97.5%
    [Cost] No protection mechanism | Time: 0ms | Memory: 0KB

  [Memory flush] Free: 94.2GB / Total: 95.0GB

Method: TOPOLOGICAL AI  (prime embedding anchors)

  Run 1/1


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

...

    [Cost] Fisher A: 2403ms | Fisher B: 2410ms | Total: 4813ms | Memory (2 tasks): 4418.4MB | Scales: GROWS WITH N
    Task A forgetting:    +6.5%  (99.5% → 93.0%)
    Task B forgetting:    +7.0%  (99.0% → 92.0%)
    Combined forgetting:  +6.7%
    Task C acc:           98.5%

  [Memory flush] Free: 94.2GB / Total: 95.0GB

Method: REPLAY  (buffer from Task A and Task B)

  Run 1/1


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A forgetting:    -0.5%  (99.5% → 100.0%)
    Task B forgetting:    -20.0%  (78.5% → 98.5%)
    Combined forgetting:  -10.2%
    Task C acc:           94.0%
    [Cost] Buffer memory: 100.0KB (A+B) | Total replay time: 259896ms | Scales: GROWS WITH N

  [Memory flush] Free: 94.2GB / Total: 95.0GB

Method: HOPE-like  (Dual Timescale EMA)

  Run 1/1


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A forgetting:    -0.5%  (99.5% → 100.0%)
    Task B forgetting:    +0.5%  (89.5% → 89.0%)
    Combined forgetting:  +0.0%
    Task C acc:           82.0%
    [Cost] EMA memory: 2262240.0KB (slow embed) | Total EMA time: 174044ms | Scales: FLAT

FINAL RESULTS — THREE TASKS  (mean ± std,  N = 1 runs)
Method              Task C Acc     Fgt A     Fgt B    Fgt Comb     Prot Time      Prot Mem
------------------------------------------------------------------------------------------
Baseline           97.5% ± 0.0   +6.5%   +3.5%    +5.0%        0ms      0.0KB
Topological        99.5% ± 0.0   +6.0%   +2.5%    +4.2%        0ms     67.5KB
EWC                98.5% ± 0.0   +6.5%   +7.0%    +6.7%     4813ms  4524480.0KB
Replay             94.0% ± 0.0   -0.5%  -20.0%   -10.2%   259896ms    100.0KB
HOPE-like          82.0% ± 0.0   -0.5%   +0.5%    +0.0%   174044ms  2262240.0KB

RANKING by Task C accuracy (higher = better):
  1. Topological       Task C: 99.5%  |  Combined forgetting: +4.2% 